# Portfolio Risk Engine

The Portfolio Risk Engine translates transaction-level BNPL model outputs into portfolio-level credit risk measures.

The analysis uses the 2024 out-of-time BNPL test population and combines:

- Portfolio-normalized Probability of Default (PD)
- Exposure at Default (EAD)
- PD × EAD exposure
- Scenario-based Loss Given Default (LGD)
- Expected Loss (EL)
- Risk-band and customer-segment concentration
- Historical macroeconomic stress context
- Management sensitivity analysis

The objective is to quantify expected credit loss and identify where portfolio risk and loss concentration are highest.

All downstream portfolio calculations use the frozen BNPL model outputs and preserve the separation between predictive modelling, portfolio aggregation and scenario analysis.

In [0]:
# Core imports
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

from functools import reduce


# -------------------------------------------------------------------
# Frozen project paths
# -------------------------------------------------------------------

BNPL_RAW_PATH = "/Volumes/workspace/default/bnpl_raw"

GOLD_BNPL_PATH = f"{BNPL_RAW_PATH}/gold_bnpl"
FINAL_TEST_PREDICTIONS_PATH = f"{BNPL_RAW_PATH}/final_bnpl_test_predictions"
FINAL_RISK_BANDS_PATH = f"{BNPL_RAW_PATH}/final_bnpl_risk_bands"
CUSTOMER_CLUSTERS_PATH = f"{BNPL_RAW_PATH}/customer_kmeans_clusters"
CUSTOMER_SEGMENTS_PATH = f"{BNPL_RAW_PATH}/final_bnpl_customer_segments"

MACRO_PATH = "/Volumes/workspace/default/macro_raw/Nigeria_Macro_Final_2022_2024.csv"


# -------------------------------------------------------------------
# Load frozen BNPL analytical inputs
# -------------------------------------------------------------------

gold_bnpl = spark.read.format("delta").load(GOLD_BNPL_PATH)
final_test_predictions = spark.read.format("delta").load(FINAL_TEST_PREDICTIONS_PATH)
final_risk_bands_existing = spark.read.format("delta").load(FINAL_RISK_BANDS_PATH)
customer_kmeans_clusters = spark.read.format("delta").load(CUSTOMER_CLUSTERS_PATH)
customer_segments = spark.read.format("delta").load(CUSTOMER_SEGMENTS_PATH)


# -------------------------------------------------------------------
# Required-column validation
# -------------------------------------------------------------------

required_gold_columns = {
    "transaction_id",
    "customer_id",
    "purchase_date",
    "principal_ngn"
}

required_prediction_columns = {
    "transaction_id",
    "customer_id",
    "purchase_date",
    "principal_ngn",
    "label",
    "default_probability",
    "risk_prediction"
}

required_existing_band_columns = {
    "transaction_id",
    "customer_id",
    "purchase_date",
    "principal_ngn",
    "label",
    "default_probability",
    "risk_prediction",
    "risk_decile",
    "risk_band"
}

required_cluster_columns = {
    "customer_id"
}

required_segment_columns = {
    "cluster",
    "segment_label"
}


def validate_required_columns(df, required_columns, dataset_name):
    missing_columns = sorted(required_columns.difference(df.columns))

    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing required columns: {missing_columns}"
        )

    return {
        "dataset": dataset_name,
        "required_columns": len(required_columns),
        "status": "PASS"
    }


input_schema_checks = [
    validate_required_columns(
        gold_bnpl,
        required_gold_columns,
        "gold_bnpl"
    ),
    validate_required_columns(
        final_test_predictions,
        required_prediction_columns,
        "final_bnpl_test_predictions"
    ),
    validate_required_columns(
        final_risk_bands_existing,
        required_existing_band_columns,
        "final_bnpl_risk_bands"
    ),
    validate_required_columns(
        customer_kmeans_clusters,
        required_cluster_columns,
        "customer_kmeans_clusters"
    ),
    validate_required_columns(
        customer_segments,
        required_segment_columns,
        "final_bnpl_customer_segments"
    )
]


display(
    spark.createDataFrame(input_schema_checks)
)


# -------------------------------------------------------------------
# Population and uniqueness audit
# -------------------------------------------------------------------

input_audit = [
    (
        "gold_bnpl",
        gold_bnpl.count(),
        gold_bnpl.select("transaction_id").distinct().count(),
        gold_bnpl.select("customer_id").distinct().count()
    ),
    (
        "final_bnpl_test_predictions",
        final_test_predictions.count(),
        final_test_predictions.select("transaction_id").distinct().count(),
        final_test_predictions.select("customer_id").distinct().count()
    ),
    (
        "final_bnpl_risk_bands_existing",
        final_risk_bands_existing.count(),
        final_risk_bands_existing.select("transaction_id").distinct().count(),
        final_risk_bands_existing.select("customer_id").distinct().count()
    ),
    (
        "customer_kmeans_clusters",
        customer_kmeans_clusters.count(),
        customer_kmeans_clusters.select("customer_id").distinct().count(),
        customer_kmeans_clusters.select("customer_id").distinct().count()
    ),
    (
        "customer_segments",
        customer_segments.count(),
        None,
        None
    )
]

input_audit_df = spark.createDataFrame(
    input_audit,
    [
        "dataset",
        "row_count",
        "distinct_transaction_id",
        "distinct_customer_id"
    ]
)

display(input_audit_df)


# -------------------------------------------------------------------
# Prediction-population integrity
# -------------------------------------------------------------------

prediction_duplicate_transactions = (
    final_test_predictions
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

prediction_null_key_count = (
    final_test_predictions
    .filter(
        F.col("transaction_id").isNull() |
        F.col("customer_id").isNull()
    )
    .count()
)

prediction_invalid_probability_count = (
    final_test_predictions
    .filter(
        F.col("default_probability").isNull() |
        (F.col("default_probability") < 0) |
        (F.col("default_probability") > 1)
    )
    .count()
)

prediction_invalid_principal_count = (
    final_test_predictions
    .filter(
        F.col("principal_ngn").isNull() |
        (F.col("principal_ngn") <= 0)
    )
    .count()
)


# -------------------------------------------------------------------
# Prediction population summary
# -------------------------------------------------------------------

prediction_population_summary = (
    final_test_predictions
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("customer_id").alias("customer_count"),
        F.sum("principal_ngn").alias("total_principal_ngn"),
        F.avg("principal_ngn").alias("average_principal_ngn"),
        F.min("principal_ngn").alias("minimum_principal_ngn"),
        F.max("principal_ngn").alias("maximum_principal_ngn"),
        F.avg("default_probability").alias("mean_raw_probability"),
        F.min("default_probability").alias("minimum_raw_probability"),
        F.max("default_probability").alias("maximum_raw_probability"),
        F.sum("label").alias("observed_default_count"),
        F.avg("label").alias("observed_default_rate")
    )
)

display(prediction_population_summary)


# -------------------------------------------------------------------
# Quality checks
# -------------------------------------------------------------------

population_quality_checks = [
    (
        "Prediction transaction uniqueness",
        prediction_duplicate_transactions == 0,
        prediction_duplicate_transactions
    ),
    (
        "Prediction key completeness",
        prediction_null_key_count == 0,
        prediction_null_key_count
    ),
    (
        "Model probability bounds",
        prediction_invalid_probability_count == 0,
        prediction_invalid_probability_count
    ),
    (
        "Principal validity",
        prediction_invalid_principal_count == 0,
        prediction_invalid_principal_count
    ),
    (
        "Prediction population matches expected OOT size",
        final_test_predictions.count() == 666246,
        final_test_predictions.count()
    )
]

population_quality_gate = spark.createDataFrame(
    population_quality_checks,
    [
        "check",
        "passed",
        "observed_value"
    ]
)

display(population_quality_gate)


failed_population_checks = (
    population_quality_gate
    .filter(~F.col("passed"))
    .count()
)

if failed_population_checks > 0:
    raise ValueError(
        f"Portfolio input quality gate failed: "
        f"{failed_population_checks} check(s) failed."
    )

print("Portfolio input quality gate: PASS")

dataset,required_columns,status
gold_bnpl,4,PASS
final_bnpl_test_predictions,7,PASS
final_bnpl_risk_bands,9,PASS
customer_kmeans_clusters,1,PASS
final_bnpl_customer_segments,2,PASS


dataset,row_count,distinct_transaction_id,distinct_customer_id
gold_bnpl,2000000,2000000,633356
final_bnpl_test_predictions,666246,666246,421457
final_bnpl_risk_bands_existing,666246,666246,421457
customer_kmeans_clusters,633356,633356,633356
customer_segments,2,null,null


transaction_count,customer_count,total_principal_ngn,average_principal_ngn,minimum_principal_ngn,maximum_principal_ngn,mean_raw_probability,minimum_raw_probability,maximum_raw_probability,observed_default_count,observed_default_rate
666246,421457,3.331791304269033E10,50008.42488013486,5000.0,500000.0,0.08065187923044928,0.018798494874795468,0.555449729124986,53310.0,0.08001548977404742


check,passed,observed_value
Prediction transaction uniqueness,true,0
Prediction key completeness,true,0
Model probability bounds,true,0
Principal validity,true,0
Prediction population matches expected OOT size,true,666246


Portfolio input quality gate: PASS


## Portfolio Probability of Default

The selected BNPL Random Forest model provides transaction-level default probabilities. These scores are used primarily for risk ranking and are not treated as fully calibrated individual probabilities of default.

Because the frozen modelling notebook did not persist validation-set predictions required for a validation-based calibration mapping, portfolio normalization is used instead.

The normalization factor aligns the mean model probability with the observed 2024 out-of-time default rate:

\[
PD_{portfolio} = \min(PD_{raw} \times \text{normalization factor}, 1)
\]

This preserves the model's relative risk ranking while ensuring that aggregate predicted default incidence is consistent with the observed out-of-time portfolio default rate.

This approach is explicitly a portfolio-level normalization, not a claim of individual-level PD calibration.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


# -------------------------------------------------------------------
# PD treatment
# -------------------------------------------------------------------

portfolio_statistics = (
    final_test_predictions
    .select(
        F.col("label").cast("double").alias("label"),
        F.col("default_probability").cast("double").alias(
            "default_probability"
        )
    )
    .agg(
        F.avg("label").alias("observed_default_rate"),
        F.avg("default_probability").alias("mean_raw_probability")
    )
    .first()
)

portfolio_default_rate = float(
    portfolio_statistics["observed_default_rate"]
)

mean_raw_probability = float(
    portfolio_statistics["mean_raw_probability"]
)

if mean_raw_probability <= 0.0:
    raise ValueError(
        "Mean raw model probability must be greater than zero."
    )

pd_normalization_factor = (
    portfolio_default_rate / mean_raw_probability
)

print(
    f"Observed portfolio default rate: "
    f"{portfolio_default_rate:.10f}"
)

print(
    f"Mean raw model probability:      "
    f"{mean_raw_probability:.10f}"
)

print(
    f"PD normalization factor:        "
    f"{pd_normalization_factor:.10f}"
)


# -------------------------------------------------------------------
# Construct portfolio-normalized model PD
# -------------------------------------------------------------------

portfolio_pd = (
    final_test_predictions
    .select(
        F.col("transaction_id"),
        F.col("customer_id"),
        F.col("purchase_date"),
        F.col("principal_ngn").cast("double").alias(
            "principal_ngn"
        ),
        F.col("label").cast("double").alias(
            "label"
        ),
        F.col("default_probability").cast("double").alias(
            "default_probability"
        ),
        F.col("risk_prediction")
    )
    .withColumn(
        "pd_normalization_factor",
        F.lit(pd_normalization_factor).cast("double")
    )
    .withColumn(
        "portfolio_pd",
        F.least(
            F.col("default_probability") *
            F.col("pd_normalization_factor"),
            F.lit(1.0).cast("double")
        )
    )
    .withColumn(
        "pd_method",
        F.lit("Portfolio-normalized model PD")
    )
)


# -------------------------------------------------------------------
# Validate PD values
# -------------------------------------------------------------------

invalid_pd_count = (
    portfolio_pd
    .filter(
        F.col("portfolio_pd").isNull() |
        (F.col("portfolio_pd") < F.lit(0.0)) |
        (F.col("portfolio_pd") > F.lit(1.0))
    )
    .count()
)

invalid_probability_count = (
    portfolio_pd
    .filter(
        F.col("default_probability").isNull() |
        (F.col("default_probability") < F.lit(0.0)) |
        (F.col("default_probability") > F.lit(1.0))
    )
    .count()
)

duplicate_transaction_count = (
    portfolio_pd
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_key_count = (
    portfolio_pd
    .filter(
        F.col("transaction_id").isNull() |
        F.col("customer_id").isNull()
    )
    .count()
)


# -------------------------------------------------------------------
# PD portfolio summary
# -------------------------------------------------------------------

pd_integrity = (
    portfolio_pd
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("transaction_id").alias(
            "distinct_transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.min("portfolio_pd").alias(
            "minimum_portfolio_pd"
        ),
        F.max("portfolio_pd").alias(
            "maximum_portfolio_pd"
        ),
        F.avg("portfolio_pd").alias(
            "mean_portfolio_pd"
        ),
        F.avg("default_probability").alias(
            "mean_raw_probability"
        ),
        F.avg("label").alias(
            "observed_default_rate"
        ),
        F.sum("portfolio_pd").alias(
            "sum_portfolio_pd"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("portfolio_pd") *
            F.col("principal_ngn")
        ).alias(
            "pd_weighted_exposure_ngn"
        )
    )
)

display(pd_integrity)


# -------------------------------------------------------------------
# PD quality gate
# -------------------------------------------------------------------

pd_summary_row = pd_integrity.first()

mean_portfolio_pd = float(
    pd_summary_row["mean_portfolio_pd"]
)

observed_default_rate = float(
    pd_summary_row["observed_default_rate"]
)

pd_quality_checks = [
    (
        "Transaction population preserved",
        bool(
            int(pd_summary_row["transaction_count"]) == 666246
        ),
        float(pd_summary_row["transaction_count"])
    ),
    (
        "Transaction IDs remain unique",
        bool(
            duplicate_transaction_count == 0
        ),
        float(duplicate_transaction_count)
    ),
    (
        "Prediction keys complete",
        bool(
            null_key_count == 0
        ),
        float(null_key_count)
    ),
    (
        "Raw model probability bounds valid",
        bool(
            invalid_probability_count == 0
        ),
        float(invalid_probability_count)
    ),
    (
        "Portfolio PD lower bound valid",
        bool(
            float(pd_summary_row["minimum_portfolio_pd"]) >= 0.0
        ),
        float(pd_summary_row["minimum_portfolio_pd"])
    ),
    (
        "Portfolio PD upper bound valid",
        bool(
            float(pd_summary_row["maximum_portfolio_pd"]) <= 1.0
        ),
        float(pd_summary_row["maximum_portfolio_pd"])
    ),
    (
        "Portfolio mean PD aligned to observed default rate",
        bool(
            abs(
                mean_portfolio_pd -
                observed_default_rate
            ) < 1e-10
        ),
        float(
            mean_portfolio_pd -
            observed_default_rate
        )
    ),
    (
        "PD normalization factor valid",
        bool(
            pd_normalization_factor > 0.0
        ),
        float(pd_normalization_factor)
    )
]


pd_quality_schema = T.StructType([
    T.StructField(
        "check",
        T.StringType(),
        False
    ),
    T.StructField(
        "passed",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "observed_value",
        T.DoubleType(),
        False
    )
])


pd_quality_gate = spark.createDataFrame(
    pd_quality_checks,
    schema=pd_quality_schema
)

display(pd_quality_gate)


failed_pd_checks = (
    pd_quality_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_pd_checks > 0:
    raise ValueError(
        f"PD construction quality gate failed: "
        f"{failed_pd_checks} check(s) failed."
    )

print("PD construction quality gate: PASS")


# -------------------------------------------------------------------
# Save authoritative portfolio PD layer
# -------------------------------------------------------------------

portfolio_pd_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_pd"
)

(
    portfolio_pd
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(portfolio_pd_output_path)
)

print(
    f"Portfolio PD layer saved to: "
    f"{portfolio_pd_output_path}"
)


# -------------------------------------------------------------------
# Calculate empirical PD distribution cutoffs
# -------------------------------------------------------------------

decile_cutoffs = portfolio_pd.approxQuantile(
    "portfolio_pd",
    [
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90
    ],
    0.0001
)

if len(decile_cutoffs) != 9:
    raise ValueError(
        "PD decile calculation did not return "
        "the expected nine percentile cutoffs."
    )

decile_cutoff_values = [
    float(value)
    for value in decile_cutoffs
]

decile_cutoff_schema = T.StructType([
    T.StructField("p10", T.DoubleType(), False),
    T.StructField("p20", T.DoubleType(), False),
    T.StructField("p30", T.DoubleType(), False),
    T.StructField("p40", T.DoubleType(), False),
    T.StructField("p50", T.DoubleType(), False),
    T.StructField("p60", T.DoubleType(), False),
    T.StructField("p70", T.DoubleType(), False),
    T.StructField("p80", T.DoubleType(), False),
    T.StructField("p90", T.DoubleType(), False)
])

decile_cutoff_df = spark.createDataFrame(
    [tuple(decile_cutoff_values)],
    schema=decile_cutoff_schema
)

display(decile_cutoff_df)


# -------------------------------------------------------------------
# Assign diagnostic PD deciles
# -------------------------------------------------------------------

p10, p20, p30, p40, p50, p60, p70, p80, p90 = (
    decile_cutoff_values
)

portfolio_pd_deciles = (
    portfolio_pd
    .withColumn(
        "risk_decile",
        F.when(
            F.col("portfolio_pd") <= F.lit(p10),
            F.lit(1)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p20),
            F.lit(2)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p30),
            F.lit(3)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p40),
            F.lit(4)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p50),
            F.lit(5)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p60),
            F.lit(6)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p70),
            F.lit(7)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p80),
            F.lit(8)
        )
        .when(
            F.col("portfolio_pd") <= F.lit(p90),
            F.lit(9)
        )
        .otherwise(
            F.lit(10)
        )
    )
)


# -------------------------------------------------------------------
# Portfolio totals
# -------------------------------------------------------------------

portfolio_totals = (
    portfolio_pd_deciles
    .agg(
        F.sum("principal_ngn").alias(
            "total_ead_ngn"
        ),
        F.sum(
            F.col("portfolio_pd") *
            F.col("principal_ngn")
        ).alias(
            "total_pd_weighted_exposure_ngn"
        )
    )
    .first()
)

total_ead_ngn = float(
    portfolio_totals["total_ead_ngn"]
)

total_pd_weighted_exposure_ngn = float(
    portfolio_totals[
        "total_pd_weighted_exposure_ngn"
    ]
)

if total_ead_ngn <= 0.0:
    raise ValueError(
        "Total EAD must be greater than zero."
    )

if total_pd_weighted_exposure_ngn <= 0.0:
    raise ValueError(
        "Total PD-weighted exposure must be "
        "greater than zero."
    )


# -------------------------------------------------------------------
# Diagnostic risk-decile summary
# -------------------------------------------------------------------

decile_summary = (
    portfolio_pd_deciles
    .groupBy("risk_decile")
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("principal_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        F.avg("label").alias(
            "observed_default_rate"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("portfolio_pd") *
            F.col("principal_ngn")
        ).alias(
            "pd_weighted_exposure_ngn"
        )
    )
    .withColumn(
        "transaction_share_pct",
        F.col("transaction_count") /
        F.lit(666246.0) *
        F.lit(100.0)
    )
    .withColumn(
        "ead_share_pct",
        F.col("ead_ngn") /
        F.lit(total_ead_ngn) *
        F.lit(100.0)
    )
    .withColumn(
        "pd_weighted_exposure_share_pct",
        F.col("pd_weighted_exposure_ngn") /
        F.lit(total_pd_weighted_exposure_ngn) *
        F.lit(100.0)
    )
    .orderBy("risk_decile")
)

display(decile_summary)


# -------------------------------------------------------------------
# Validate decile assignment
# -------------------------------------------------------------------

invalid_decile_count = (
    portfolio_pd_deciles
    .filter(
        F.col("risk_decile").isNull() |
        (F.col("risk_decile") < 1) |
        (F.col("risk_decile") > 10)
    )
    .count()
)

decile_population_count = (
    portfolio_pd_deciles
    .count()
)

if invalid_decile_count > 0:
    raise ValueError(
        f"Invalid risk-decile assignments found: "
        f"{invalid_decile_count}"
    )

if decile_population_count != 666246:
    raise ValueError(
        "Risk-decile assignment changed the "
        "transaction population."
    )

print(
    "PD diagnostic decile assignment: PASS"
)


# -------------------------------------------------------------------
# Save diagnostic PD layer
# -------------------------------------------------------------------

pd_diagnostic_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_pd_diagnostic"
)

(
    portfolio_pd_deciles
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(pd_diagnostic_output_path)
)

print(
    f"PD diagnostic layer saved to: "
    f"{pd_diagnostic_output_path}"
)

Observed portfolio default rate: 0.0800154898
Mean raw model probability:      0.0806518792
PD normalization factor:        0.9921094280


transaction_count,distinct_transaction_count,customer_count,minimum_portfolio_pd,maximum_portfolio_pd,mean_portfolio_pd,mean_raw_probability,observed_default_rate,sum_portfolio_pd,observed_default_count,pd_weighted_exposure_ngn
666246,666246,421457,0.018650163998333667,0.5510669130697472,0.08001548977404752,0.080651879230448,0.08001548977404742,53310.00000000006,53310.0,3.289674922513501E9


check,passed,observed_value
Transaction population preserved,true,666246.0
Transaction IDs remain unique,true,0.0
Prediction keys complete,true,0.0
Raw model probability bounds valid,true,0.0
Portfolio PD lower bound valid,true,0.018650163998333667
Portfolio PD upper bound valid,true,0.5510669130697472
Portfolio mean PD aligned to observed default rate,true,9.71445146547012E-17
PD normalization factor valid,true,0.9921094280446527


PD construction quality gate: PASS
Portfolio PD layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_pd


p10,p20,p30,p40,p50,p60,p70,p80,p90
0.02378285014947668,0.02479532774537294,0.03881699562064688,0.04245067545290218,0.046784466337159714,0.07061971030157395,0.0749397207624142,0.12854423540558782,0.18295185589995275


risk_decile,transaction_count,customer_count,ead_ngn,average_pd,observed_default_rate,observed_default_count,pd_weighted_exposure_ngn,transaction_share_pct,ead_share_pct,pd_weighted_exposure_share_pct
1,66655,63646,2.4938496308413787E9,0.02299886679962705,0.0,0.0,5.735869801622548E7,10.004562879176762,7.485011524119408,1.7435977525828033
2,66559,63426,2.503093149836094E9,0.024277614157849497,0.0,0.0,6.07694089961505E7,9.990153787039622,7.512754915438282,1.847277023643393
3,66656,62973,2.546655217970072E9,0.028525867957312448,0.0,0.0,7.28779835707913E7,10.004712973886521,7.64350160439855,2.2153551730001433
4,66640,63785,2.764335210749634E9,0.04089649110630947,0.0,0.0,1.1308595074945927E8,10.002311458530333,8.296843824544977,3.437602602480098
5,66617,63355,3.373399517836947E9,0.04413358030557855,0.0,0.0,1.4950634369576955E8,9.99885928020581,10.12488241239691,4.544714818859308
6,66589,62843,4.786789865515949E9,0.05931575957645362,5.25612338374206E-4,35.0,2.7633666258905506E8,9.994656628332478,14.367015903375366,8.400120653195664
7,66684,63710,2.929755635691237E9,0.07285408785370813,5.998440405494571E-5,4.0,2.133092739760148E8,10.008915625759855,8.793334780415035,6.484205248250918
8,66606,62769,3.5330110752302628E9,0.08470510272268433,0.022100111101102004,1472.0,3.024965256980727E8,9.997208238398429,10.603938700192638,9.195331843516863
9,66633,62913,2.5771995262387114E9,0.17061535093018834,0.31254783665751207,20826.0,4.399116117417481E8,10.001260795561999,7.735176939013609,13.372494915261424
10,66607,63673,5.809824212780002E9,0.2518631346897963,0.46501118501058447,30973.0,1.6040224634802513E9,9.997358333108192,17.43753939610814,48.75929996921051


PD diagnostic decile assignment: PASS
PD diagnostic layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_pd_diagnostic


## Empirical Portfolio Risk Bands

Risk bands are constructed from the empirical distribution of portfolio-normalized PD rather than imposing arbitrary PD cutoffs in advance.

Transactions are first ranked into ten PD deciles. The final five risk bands consolidate these deciles into:

- Very Low: Deciles 1–2
- Low: Deciles 3–4
- Moderate: Deciles 5–6
- High: Deciles 7–8
- Very High: Deciles 9–10

The band structure is therefore determined from the observed portfolio risk distribution and subsequently assessed using exposure concentration and expected-loss contribution.

The risk bands are used as an interpretation and portfolio-management layer, rather than as additional predictive-model inputs.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


# -------------------------------------------------------------------
# Authoritative portfolio risk-band construction
# -------------------------------------------------------------------

portfolio_risk_bands = (
    portfolio_pd_deciles
    .withColumn(
        "risk_band",
        F.when(
            F.col("risk_decile").isin(1, 2),
            F.lit("Very Low")
        )
        .when(
            F.col("risk_decile").isin(3, 4),
            F.lit("Low")
        )
        .when(
            F.col("risk_decile").isin(5, 6),
            F.lit("Moderate")
        )
        .when(
            F.col("risk_decile").isin(7, 8),
            F.lit("High")
        )
        .when(
            F.col("risk_decile").isin(9, 10),
            F.lit("Very High")
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )
    .withColumn(
        "risk_band_order",
        F.when(
            F.col("risk_band") == "Very Low",
            F.lit(1)
        )
        .when(
            F.col("risk_band") == "Low",
            F.lit(2)
        )
        .when(
            F.col("risk_band") == "Moderate",
            F.lit(3)
        )
        .when(
            F.col("risk_band") == "High",
            F.lit(4)
        )
        .when(
            F.col("risk_band") == "Very High",
            F.lit(5)
        )
        .otherwise(
            F.lit(None).cast("int")
        )
    )
)


# -------------------------------------------------------------------
# Band assignment validation
# -------------------------------------------------------------------

invalid_band_count = (
    portfolio_risk_bands
    .filter(
        F.col("risk_band").isNull() |
        F.col("risk_band_order").isNull()
    )
    .count()
)

invalid_decile_count = (
    portfolio_risk_bands
    .filter(
        F.col("risk_decile").isNull() |
        (F.col("risk_decile") < 1) |
        (F.col("risk_decile") > 10)
    )
    .count()
)

band_population_count = (
    portfolio_risk_bands.count()
)

expected_band_count = (
    portfolio_risk_bands
    .select("risk_band")
    .distinct()
    .count()
)


band_quality_checks = [
    (
        "Transaction population preserved",
        bool(band_population_count == 666246),
        float(band_population_count)
    ),
    (
        "All transactions have valid risk bands",
        bool(invalid_band_count == 0),
        float(invalid_band_count)
    ),
    (
        "All transactions have valid risk deciles",
        bool(invalid_decile_count == 0),
        float(invalid_decile_count)
    ),
    (
        "Exactly five risk bands created",
        bool(expected_band_count == 5),
        float(expected_band_count)
    )
]


band_quality_schema = T.StructType([
    T.StructField(
        "check",
        T.StringType(),
        False
    ),
    T.StructField(
        "passed",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "observed_value",
        T.DoubleType(),
        False
    )
])


band_quality_gate = spark.createDataFrame(
    band_quality_checks,
    schema=band_quality_schema
)

display(band_quality_gate)


failed_band_checks = (
    band_quality_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_band_checks > 0:
    raise ValueError(
        f"Risk-band quality gate failed: "
        f"{failed_band_checks} check(s) failed."
    )

print("Risk-band assignment quality gate: PASS")


# -------------------------------------------------------------------
# Portfolio totals for aggregation
# -------------------------------------------------------------------

band_portfolio_totals = (
    portfolio_risk_bands
    .agg(
        F.sum("principal_ngn").alias(
            "total_ead_ngn"
        ),
        F.sum(
            F.col("portfolio_pd") *
            F.col("principal_ngn")
        ).alias(
            "total_pd_weighted_exposure_ngn"
        ),
        F.sum("label").alias(
            "total_observed_defaults"
        )
    )
    .first()
)

band_total_ead = float(
    band_portfolio_totals["total_ead_ngn"]
)

band_total_pd_weighted_exposure = float(
    band_portfolio_totals[
        "total_pd_weighted_exposure_ngn"
    ]
)

band_total_observed_defaults = float(
    band_portfolio_totals[
        "total_observed_defaults"
    ]
)


# -------------------------------------------------------------------
# Risk-band portfolio summary
# -------------------------------------------------------------------

risk_band_summary = (
    portfolio_risk_bands
    .groupBy(
        "risk_band",
        "risk_band_order"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("principal_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        F.min("portfolio_pd").alias(
            "minimum_pd"
        ),
        F.max("portfolio_pd").alias(
            "maximum_pd"
        ),
        F.avg("label").alias(
            "observed_default_rate"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("portfolio_pd") *
            F.col("principal_ngn")
        ).alias(
            "pd_weighted_exposure_ngn"
        ),
        F.sum(
            F.col("label") *
            F.col("principal_ngn")
        ).alias(
            "observed_default_exposure_ngn"
        )
    )
    .withColumn(
        "transaction_share_pct",
        F.col("transaction_count") /
        F.lit(666246.0) *
        F.lit(100.0)
    )
    .withColumn(
        "ead_share_pct",
        F.col("ead_ngn") /
        F.lit(band_total_ead) *
        F.lit(100.0)
    )
    .withColumn(
        "pd_weighted_exposure_share_pct",
        F.col("pd_weighted_exposure_ngn") /
        F.lit(band_total_pd_weighted_exposure) *
        F.lit(100.0)
    )
    .withColumn(
        "observed_default_exposure_share_pct",
        F.col("observed_default_exposure_ngn") /
        F.sum("observed_default_exposure_ngn").over(
            Window.partitionBy()
        ) *
        F.lit(100.0)
    )
    .orderBy("risk_band_order")
)


display(risk_band_summary)


# -------------------------------------------------------------------
# Reconciliation checks
# -------------------------------------------------------------------

risk_band_reconciliation = (
    risk_band_summary
    .agg(
        F.sum("transaction_count").alias(
            "band_transaction_count"
        ),
        F.sum("ead_ngn").alias(
            "band_ead_ngn"
        ),
        F.sum("pd_weighted_exposure_ngn").alias(
            "band_pd_weighted_exposure_ngn"
        ),
        F.sum("observed_default_count").alias(
            "band_observed_default_count"
        ),
        F.sum("observed_default_exposure_ngn").alias(
            "band_observed_default_exposure_ngn"
        )
    )
    .first()
)


reconciliation_checks = [
    (
        "Risk-band transaction count reconciles",
        bool(
            int(
                risk_band_reconciliation[
                    "band_transaction_count"
                ]
            ) == 666246
        ),
        float(
            risk_band_reconciliation[
                "band_transaction_count"
            ]
        )
    ),
    (
        "Risk-band EAD reconciles",
        bool(
            abs(
                float(
                    risk_band_reconciliation[
                        "band_ead_ngn"
                    ]
                ) -
                band_total_ead
            ) < 0.01
        ),
        float(
            risk_band_reconciliation[
                "band_ead_ngn"
            ] -
            band_total_ead
        )
    ),
    (
        "Risk-band PD-weighted exposure reconciles",
        bool(
            abs(
                float(
                    risk_band_reconciliation[
                        "band_pd_weighted_exposure_ngn"
                    ]
                ) -
                band_total_pd_weighted_exposure
            ) < 0.01
        ),
        float(
            risk_band_reconciliation[
                "band_pd_weighted_exposure_ngn"
            ] -
            band_total_pd_weighted_exposure
        )
    ),
    (
        "Risk-band observed defaults reconcile",
        bool(
            float(
                risk_band_reconciliation[
                    "band_observed_default_count"
                ]
            ) == band_total_observed_defaults
        ),
        float(
            risk_band_reconciliation[
                "band_observed_default_count"
            ]
        )
    )
]


reconciliation_schema = T.StructType([
    T.StructField(
        "check",
        T.StringType(),
        False
    ),
    T.StructField(
        "passed",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "observed_value",
        T.DoubleType(),
        False
    )
])


risk_band_reconciliation_gate = spark.createDataFrame(
    reconciliation_checks,
    schema=reconciliation_schema
)

display(risk_band_reconciliation_gate)


failed_reconciliation_checks = (
    risk_band_reconciliation_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_reconciliation_checks > 0:
    raise ValueError(
        f"Risk-band reconciliation failed: "
        f"{failed_reconciliation_checks} check(s) failed."
    )

print("Risk-band reconciliation: PASS")


# -------------------------------------------------------------------
# Persist authoritative portfolio risk-band layer
# -------------------------------------------------------------------

portfolio_risk_bands_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk_bands"
)

(
    portfolio_risk_bands
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        portfolio_risk_bands_output_path
    )
)

print(
    f"Portfolio risk-band layer saved to: "
    f"{portfolio_risk_bands_output_path}"
)


# -------------------------------------------------------------------
# Persist risk-band summary
# -------------------------------------------------------------------

risk_band_summary_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk_band_summary"
)

(
    risk_band_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        risk_band_summary_output_path
    )
)

print(
    f"Risk-band summary saved to: "
    f"{risk_band_summary_output_path}"
)

check,passed,observed_value
Transaction population preserved,true,666246.0
All transactions have valid risk bands,true,0.0
All transactions have valid risk deciles,true,0.0
Exactly five risk bands created,true,5.0


Risk-band assignment quality gate: PASS


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


risk_band,risk_band_order,transaction_count,customer_count,ead_ngn,average_pd,minimum_pd,maximum_pd,observed_default_rate,observed_default_count,pd_weighted_exposure_ngn,observed_default_exposure_ngn,transaction_share_pct,ead_share_pct,pd_weighted_exposure_share_pct,observed_default_exposure_share_pct
Very Low,1,133214,121278,4.996942780677462E9,0.023637779717307888,0.018650163998333667,0.02479532774537294,0.0,0.0,1.1812810701237577E8,0.0,19.994716666216384,14.997766439557658,3.59087477622619,0.0
Low,2,133296,120782,5.310990428719698E9,0.03471043708653736,0.024795363277146824,0.04245067545290218,0.0,0.0,1.8596393432025048E8,0.0,20.007024432416856,15.940345428943504,5.652957775480239,0.0
Moderate,3,133206,119963,8.160189383352885E9,0.051723074288344176,0.04245080434443494,0.07061971030157395,2.627509271354143E-4,35.0,4.258430062848243E8,1322145.1858605405,19.99351590853829,24.491898315772243,12.944835472054963,0.03426092963809923
High,4,133290,120632,6.462766710921511E9,0.07877612773939378,0.07061977240037016,0.12854423540558782,0.011073598919648885,1476.0,5.1580579967408717E8,7.267217819862382E7,20.006123864158283,19.39727348060771,15.679537091767774,1.8831641264041064
Very High,5,133240,120673,8.387023739018707E9,0.21123131560202968,0.1285454971345337,0.5510669130697472,0.38876463524467125,51799.0,2.043934075222002E9,3.7850521176387434E9,19.998619128670192,25.17271633512173,62.131794884472015,98.0825749439578


check,passed,observed_value
Risk-band transaction count reconciles,true,666246.0
Risk-band EAD reconciles,true,9.72747802734375E-4
Risk-band PD-weighted exposure reconciles,true,2.86102294921875E-5
Risk-band observed defaults reconcile,true,53310.0


Risk-band reconciliation: PASS
Portfolio risk-band layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_bands
Risk-band summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_band_summary


## Exposure at Default and Portfolio Risk Exposure

The BNPL dataset does not provide a sufficiently reliable undrawn commitment or recovery structure to support a more complex EAD model.

Therefore, principal amount is used as a transparent EAD proxy:

\[
EAD = Principal
\]

The portfolio risk exposure is then calculated as:

\[
PD * EAD
\]

This provides an exposure-weighted view of default risk and allows portfolio concentration to be assessed beyond transaction counts alone

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


# -------------------------------------------------------------------
# EAD construction
# -------------------------------------------------------------------

portfolio_ead = (
    portfolio_risk_bands
    .select(
        F.col("transaction_id"),
        F.col("customer_id"),
        F.col("purchase_date"),
        F.col("principal_ngn").cast("double").alias(
            "principal_ngn"
        ),
        F.col("label").cast("double").alias(
            "label"
        ),
        F.col("default_probability").cast("double").alias(
            "default_probability"
        ),
        F.col("portfolio_pd").cast("double").alias(
            "portfolio_pd"
        ),
        F.col("pd_normalization_factor").cast("double").alias(
            "pd_normalization_factor"
        ),
        F.col("pd_method"),
        F.col("risk_decile").cast("int").alias(
            "risk_decile"
        ),
        F.col("risk_band"),
        F.col("risk_band_order").cast("int").alias(
            "risk_band_order"
        )
    )
    .withColumn(
        "ead_ngn",
        F.col("principal_ngn").cast("double")
    )
    .withColumn(
        "ead_conversion_factor",
        F.lit(1.0).cast("double")
    )
    .withColumn(
        "ead_method",
        F.lit("Principal-based EAD proxy")
    )
)


# -------------------------------------------------------------------
# EAD integrity checks
# -------------------------------------------------------------------

ead_invalid_count = (
    portfolio_ead
    .filter(
        F.col("ead_ngn").isNull() |
        (F.col("ead_ngn") <= F.lit(0.0))
    )
    .count()
)

ead_conversion_invalid_count = (
    portfolio_ead
    .filter(
        F.col("ead_conversion_factor").isNull() |
        (F.col("ead_conversion_factor") <= F.lit(0.0))
    )
    .count()
)

ead_principal_mismatch_count = (
    portfolio_ead
    .filter(
        F.abs(
            F.col("ead_ngn") -
            F.col("principal_ngn")
        ) > F.lit(1e-8)
    )
    .count()
)

ead_null_key_count = (
    portfolio_ead
    .filter(
        F.col("transaction_id").isNull() |
        F.col("customer_id").isNull()
    )
    .count()
)

ead_duplicate_transaction_count = (
    portfolio_ead
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


# -------------------------------------------------------------------
# EAD portfolio summary
# -------------------------------------------------------------------

ead_portfolio_summary = (
    portfolio_ead
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("transaction_id").alias(
            "distinct_transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "total_ead_ngn"
        ),
        F.avg("ead_ngn").alias(
            "average_ead_ngn"
        ),
        F.min("ead_ngn").alias(
            "minimum_ead_ngn"
        ),
        F.max("ead_ngn").alias(
            "maximum_ead_ngn"
        )
    )
)

display(ead_portfolio_summary)


# -------------------------------------------------------------------
# EAD quality gate
# -------------------------------------------------------------------

ead_summary_row = ead_portfolio_summary.first()

ead_quality_checks = [
    (
        "Transaction population preserved",
        bool(
            int(ead_summary_row["transaction_count"]) == 666246
        ),
        float(ead_summary_row["transaction_count"])
    ),
    (
        "Transaction IDs remain unique",
        bool(
            int(ead_summary_row["distinct_transaction_count"]) == 666246
        ),
        float(ead_summary_row["distinct_transaction_count"])
    ),
    (
        "EAD values are positive and non-null",
        bool(
            ead_invalid_count == 0
        ),
        float(ead_invalid_count)
    ),
    (
        "EAD conversion factor is valid",
        bool(
            ead_conversion_invalid_count == 0
        ),
        float(ead_conversion_invalid_count)
    ),
    (
        "EAD equals principal under selected methodology",
        bool(
            ead_principal_mismatch_count == 0
        ),
        float(ead_principal_mismatch_count)
    ),
    (
        "EAD keys are complete",
        bool(
            ead_null_key_count == 0
        ),
        float(ead_null_key_count)
    ),
    (
        "EAD transactions are unique",
        bool(
            ead_duplicate_transaction_count == 0
        ),
        float(ead_duplicate_transaction_count)
    )
]


ead_quality_schema = T.StructType([
    T.StructField(
        "check",
        T.StringType(),
        False
    ),
    T.StructField(
        "passed",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "observed_value",
        T.DoubleType(),
        False
    )
])


ead_quality_gate = spark.createDataFrame(
    ead_quality_checks,
    schema=ead_quality_schema
)

display(ead_quality_gate)


failed_ead_checks = (
    ead_quality_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_ead_checks > 0:
    raise ValueError(
        f"EAD quality gate failed: "
        f"{failed_ead_checks} check(s) failed."
    )

print("EAD construction quality gate: PASS")


# -------------------------------------------------------------------
# PD × EAD risk exposure
# -------------------------------------------------------------------

portfolio_risk = (
    portfolio_ead
    .withColumn(
        "pd_ead_loss_exposure_ngn",
        F.col("portfolio_pd") *
        F.col("ead_ngn")
    )
)


# -------------------------------------------------------------------
# PD × EAD integrity checks
# -------------------------------------------------------------------

invalid_pd_ead_count = (
    portfolio_risk
    .filter(
        F.col("pd_ead_loss_exposure_ngn").isNull() |
        (F.col("pd_ead_loss_exposure_ngn") < F.lit(0.0))
    )
    .count()
)

pd_ead_recalculated_mismatch_count = (
    portfolio_risk
    .filter(
        F.abs(
            F.col("pd_ead_loss_exposure_ngn") -
            (
                F.col("portfolio_pd") *
                F.col("ead_ngn")
            )
        ) > F.lit(1e-8)
    )
    .count()
)


# -------------------------------------------------------------------
# Portfolio risk summary
# -------------------------------------------------------------------

portfolio_risk_summary = (
    portfolio_risk
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "total_ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "transaction_weighted_pd"
        ),
        (
            F.sum(
                F.col("portfolio_pd") *
                F.col("ead_ngn")
            ) /
            F.sum("ead_ngn")
        ).alias(
            "exposure_weighted_pd"
        ),
        F.sum(
            "pd_ead_loss_exposure_ngn"
        ).alias(
            "total_pd_ead_loss_exposure_ngn"
        ),
        F.sum(
            F.col("label") *
            F.col("ead_ngn")
        ).alias(
            "observed_default_exposure_ngn"
        ),
        F.sum("label").alias(
            "observed_default_count"
        )
    )
)

display(portfolio_risk_summary)


# -------------------------------------------------------------------
# PD × EAD quality gate
# -------------------------------------------------------------------

portfolio_risk_summary_row = (
    portfolio_risk_summary.first()
)

risk_population_count = int(
    portfolio_risk_summary_row[
        "transaction_count"
    ]
)

risk_total_ead = float(
    portfolio_risk_summary_row[
        "total_ead_ngn"
    ]
)

risk_pd_ead_exposure = float(
    portfolio_risk_summary_row[
        "total_pd_ead_loss_exposure_ngn"
    ]
)

risk_observed_default_count = float(
    portfolio_risk_summary_row[
        "observed_default_count"
    ]
)

pd_ead_quality_checks = [
    (
        "Transaction population preserved",
        bool(
            risk_population_count == 666246
        ),
        float(risk_population_count)
    ),
    (
        "Total EAD is positive",
        bool(
            risk_total_ead > 0.0
        ),
        risk_total_ead
    ),
    (
        "PD × EAD exposure is non-negative",
        bool(
            invalid_pd_ead_count == 0
        ),
        float(invalid_pd_ead_count)
    ),
    (
        "PD × EAD formula integrity",
        bool(
            pd_ead_recalculated_mismatch_count == 0
        ),
        float(pd_ead_recalculated_mismatch_count)
    ),
    (
        "Observed default count preserved",
        bool(
            risk_observed_default_count == 53310.0
        ),
        risk_observed_default_count
    )
]


pd_ead_quality_gate = spark.createDataFrame(
    pd_ead_quality_checks,
    schema=ead_quality_schema
)

display(pd_ead_quality_gate)


failed_pd_ead_checks = (
    pd_ead_quality_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_pd_ead_checks > 0:
    raise ValueError(
        f"PD × EAD quality gate failed: "
        f"{failed_pd_ead_checks} check(s) failed."
    )

print("PD × EAD risk-engine quality gate: PASS")


# -------------------------------------------------------------------
# Risk-band aggregation
# -------------------------------------------------------------------

risk_band_portfolio_summary = (
    portfolio_risk
    .groupBy(
        "risk_band",
        "risk_band_order"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        (
            F.sum(
                F.col("portfolio_pd") *
                F.col("ead_ngn")
            ) /
            F.sum("ead_ngn")
        ).alias(
            "exposure_weighted_pd"
        ),
        F.sum(
            "pd_ead_loss_exposure_ngn"
        ).alias(
            "pd_weighted_exposure_ngn"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("label") *
            F.col("ead_ngn")
        ).alias(
            "observed_default_exposure_ngn"
        )
    )
    .withColumn(
        "ead_share_pct",
        F.col("ead_ngn") /
        F.lit(risk_total_ead) *
        F.lit(100.0)
    )
    .withColumn(
        "pd_weighted_exposure_share_pct",
        F.col("pd_weighted_exposure_ngn") /
        F.lit(risk_pd_ead_exposure) *
        F.lit(100.0)
    )
    .orderBy("risk_band_order")
)

display(risk_band_portfolio_summary)


# -------------------------------------------------------------------
# Persist EAD layer
# -------------------------------------------------------------------

portfolio_ead_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_ead"
)

(
    portfolio_ead
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(portfolio_ead_output_path)
)

print(
    f"EAD layer saved to: "
    f"{portfolio_ead_output_path}"
)


# -------------------------------------------------------------------
# Persist PD × EAD risk layer
# -------------------------------------------------------------------

portfolio_risk_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk"
)

(
    portfolio_risk
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(portfolio_risk_output_path)
)

print(
    f"Portfolio risk layer saved to: "
    f"{portfolio_risk_output_path}"
)


# -------------------------------------------------------------------
# Persist risk-band portfolio summary
# -------------------------------------------------------------------

risk_band_portfolio_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_portfolio_risk_band_summary"
)

(
    risk_band_portfolio_summary
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(risk_band_portfolio_output_path)
)

print(
    f"Risk-band portfolio summary saved to: "
    f"{risk_band_portfolio_output_path}"
)

transaction_count,distinct_transaction_count,customer_count,total_ead_ngn,average_ead_ngn,minimum_ead_ngn,maximum_ead_ngn
666246,666246,421457,3.3317913042689316E10,50008.42488013334,5000.0,500000.0


check,passed,observed_value
Transaction population preserved,true,666246.0
Transaction IDs remain unique,true,666246.0
EAD values are positive and non-null,true,0.0
EAD conversion factor is valid,true,0.0
EAD equals principal under selected methodology,true,0.0
EAD keys are complete,true,0.0
EAD transactions are unique,true,0.0


EAD construction quality gate: PASS


transaction_count,customer_count,total_ead_ngn,transaction_weighted_pd,exposure_weighted_pd,total_pd_ead_loss_exposure_ngn,observed_default_exposure_ngn,observed_default_count
666246,421457,3.331791304269033E10,0.08001548977404745,0.09873592377465126,3.2896749225135317E9,3.859046441023226E9,53310.0


check,passed,observed_value
Transaction population preserved,true,666246.0
Total EAD is positive,true,3.331791304269033E10
PD × EAD exposure is non-negative,true,0.0
PD × EAD formula integrity,true,0.0
Observed default count preserved,true,53310.0


PD × EAD risk-engine quality gate: PASS


risk_band,risk_band_order,transaction_count,customer_count,ead_ngn,average_pd,exposure_weighted_pd,pd_weighted_exposure_ngn,observed_default_count,observed_default_exposure_ngn,ead_share_pct,pd_weighted_exposure_share_pct
Very Low,1,133214,121278,4.996942780677462E9,0.023637779717307888,0.023640075981890777,1.1812810701237577E8,0.0,0.0,14.9977664395572,3.5908747762261566
Low,2,133296,120782,5.310990428719698E9,0.03471043708653736,0.03501492552399123,1.8596393432025048E8,0.0,0.0,15.940345428943017,5.6529577754801865
Moderate,3,133206,119963,8.160189383352885E9,0.051723074288344176,0.052185431768723554,4.258430062848243E8,35.0,1322145.1858605405,24.4918983157715,12.94483547205484
High,4,133290,120632,6.462766710921511E9,0.07877612773939378,0.0798119168996802,5.1580579967408717E8,1476.0,7.267217819862382E7,19.397273480607115,15.679537091767628
Very High,5,133240,120673,8.387023739018707E9,0.21123131560202968,0.2437019542120844,2.043934075222002E9,51799.0,3.7850521176387434E9,25.172716335120963,62.13179488447144


EAD layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_ead
Portfolio risk layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk
Risk-band portfolio summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_band_summary


## Scenario-Based LGD and Expected Loss

The BNPL dataset does not contain a reliable recovery field that supports empirical LGD estimation. LGD is therefore treated as a scenario assumption rather than a statistically estimated parameter.

Three severity assumptions are evaluated:

- Baseline LGD: 40%
- Moderate LGD Sensitivity: 55%
- Severe LGD Sensitivity: 70%

Expected Loss is calculated as:

\[
EL = PD * EAD * LGD
\]

The sensitivity scenarios quantify the effect of alternative loss-severity assumptions while avoiding unsupported claims about observed recovery behaviour.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T


# -------------------------------------------------------------------
# LGD scenario framework
# -------------------------------------------------------------------

LGD_SCENARIOS = [
    (
        "Baseline",
        0.40,
        "Base severity assumption"
    ),
    (
        "Moderate LGD Sensitivity",
        0.55,
        "Higher loss severity sensitivity"
    ),
    (
        "Severe LGD Sensitivity",
        0.70,
        "Severe loss severity sensitivity"
    )
]


lgd_schema = T.StructType([
    T.StructField(
        "lgd_scenario",
        T.StringType(),
        False
    ),
    T.StructField(
        "lgd",
        T.DoubleType(),
        False
    ),
    T.StructField(
        "lgd_description",
        T.StringType(),
        False
    )
])


lgd_scenario_df = spark.createDataFrame(
    LGD_SCENARIOS,
    schema=lgd_schema
)


# -------------------------------------------------------------------
# Validate LGD scenario definitions
# -------------------------------------------------------------------

invalid_lgd_count = (
    lgd_scenario_df
    .filter(
        F.col("lgd").isNull() |
        (F.col("lgd") < F.lit(0.0)) |
        (F.col("lgd") > F.lit(1.0))
    )
    .count()
)

scenario_count = (
    lgd_scenario_df.count()
)

distinct_scenario_count = (
    lgd_scenario_df
    .select("lgd_scenario")
    .distinct()
    .count()
)

lgd_quality_checks = [
    (
        "Exactly three LGD scenarios defined",
        bool(scenario_count == 3),
        float(scenario_count)
    ),
    (
        "LGD scenario names are unique",
        bool(distinct_scenario_count == 3),
        float(distinct_scenario_count)
    ),
    (
        "All LGD values are within [0, 1]",
        bool(invalid_lgd_count == 0),
        float(invalid_lgd_count)
    )
]


lgd_quality_schema = T.StructType([
    T.StructField(
        "check",
        T.StringType(),
        False
    ),
    T.StructField(
        "passed",
        T.BooleanType(),
        False
    ),
    T.StructField(
        "observed_value",
        T.DoubleType(),
        False
    )
])


lgd_quality_gate = spark.createDataFrame(
    lgd_quality_checks,
    schema=lgd_quality_schema
)

display(lgd_quality_gate)


failed_lgd_checks = (
    lgd_quality_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_lgd_checks > 0:
    raise ValueError(
        f"LGD scenario quality gate failed: "
        f"{failed_lgd_checks} check(s) failed."
    )

print("LGD scenario quality gate: PASS")


# -------------------------------------------------------------------
# Construct transaction-level expected loss
# -------------------------------------------------------------------

expected_loss = (
    portfolio_risk
    .crossJoin(
        F.broadcast(lgd_scenario_df)
    )
    .withColumn(
        "expected_loss_ngn",
        F.col("portfolio_pd") *
        F.col("ead_ngn") *
        F.col("lgd")
    )
    .withColumn(
        "pd_ead_exposure_ngn",
        F.col("portfolio_pd") *
        F.col("ead_ngn")
    )
    .withColumn(
        "expected_loss_method",
        F.lit(
            "Simplified PD x EAD x scenario LGD"
        )
    )
)


# -------------------------------------------------------------------
# Transaction-level EL integrity checks
# -------------------------------------------------------------------

invalid_expected_loss_count = (
    expected_loss
    .filter(
        F.col("expected_loss_ngn").isNull() |
        (F.col("expected_loss_ngn") < F.lit(0.0))
    )
    .count()
)


el_formula_mismatch_count = (
    expected_loss
    .filter(
        F.abs(
            F.col("expected_loss_ngn") -
            (
                F.col("portfolio_pd") *
                F.col("ead_ngn") *
                F.col("lgd")
            )
        ) > F.lit(1e-8)
    )
    .count()
)


expected_scenario_row_count = (
    666246 * 3
)

actual_scenario_row_count = (
    expected_loss.count()
)

distinct_transaction_scenario_count = (
    expected_loss
    .select(
        "transaction_id",
        "lgd_scenario"
    )
    .distinct()
    .count()
)


# -------------------------------------------------------------------
# Expected-loss transaction integrity gate
# -------------------------------------------------------------------

el_integrity_checks = [
    (
        "Expected-loss row count",
        bool(
            actual_scenario_row_count ==
            expected_scenario_row_count
        ),
        float(actual_scenario_row_count)
    ),
    (
        "Three scenario observations per transaction",
        bool(
            distinct_transaction_scenario_count ==
            expected_scenario_row_count
        ),
        float(distinct_transaction_scenario_count)
    ),
    (
        "Expected loss is non-negative and non-null",
        bool(
            invalid_expected_loss_count == 0
        ),
        float(invalid_expected_loss_count)
    ),
    (
        "Transaction-level EL formula integrity",
        bool(
            el_formula_mismatch_count == 0
        ),
        float(el_formula_mismatch_count)
    )
]


el_integrity_gate = spark.createDataFrame(
    el_integrity_checks,
    schema=lgd_quality_schema
)

display(el_integrity_gate)


failed_el_integrity_checks = (
    el_integrity_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_el_integrity_checks > 0:
    raise ValueError(
        f"Expected-loss integrity gate failed: "
        f"{failed_el_integrity_checks} check(s) failed."
    )

print("Transaction-level expected-loss integrity: PASS")


# -------------------------------------------------------------------
# Portfolio expected loss by LGD scenario
# -------------------------------------------------------------------

portfolio_expected_loss = (
    expected_loss
    .groupBy(
        "lgd_scenario",
        "lgd",
        "lgd_description"
    )
    .agg(
        F.count("*").alias(
            "scenario_transaction_rows"
        ),
        F.countDistinct("transaction_id").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "total_ead_ngn"
        ),
        F.sum(
            "pd_ead_exposure_ngn"
        ).alias(
            "pd_ead_exposure_ngn"
        ),
        F.sum(
            "expected_loss_ngn"
        ).alias(
            "expected_loss_ngn"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("label") *
            F.col("ead_ngn")
        ).alias(
            "observed_default_exposure_ngn"
        )
    )
    .withColumn(
        "expected_loss_rate_of_ead",
        F.col("expected_loss_ngn") /
        F.col("total_ead_ngn")
    )
    .orderBy(
        F.when(
            F.col("lgd_scenario") == "Baseline",
            1
        )
        .when(
            F.col("lgd_scenario") ==
            "Moderate LGD Sensitivity",
            2
        )
        .otherwise(3)
    )
)


display(portfolio_expected_loss)


# -------------------------------------------------------------------
# Independent EL reconciliation
# -------------------------------------------------------------------

baseline_el_from_transaction_sum = (
    expected_loss
    .filter(
        F.col("lgd_scenario") ==
        F.lit("Baseline")
    )
    .agg(
        F.sum("expected_loss_ngn").alias(
            "transaction_sum_el"
        )
    )
    .first()["transaction_sum_el"]
)

baseline_el_from_portfolio_formula = (
    expected_loss
    .filter(
        F.col("lgd_scenario") ==
        F.lit("Baseline")
    )
    .agg(
        (
            F.sum("portfolio_pd") *
            F.lit(0.0)
        ).alias("placeholder")
    )
)

baseline_components = (
    portfolio_risk
    .agg(
        F.sum(
            F.col("portfolio_pd") *
            F.col("ead_ngn")
        ).alias(
            "total_pd_ead"
        ),
        F.sum("ead_ngn").alias(
            "total_ead"
        )
    )
    .first()
)

total_pd_ead = float(
    baseline_components["total_pd_ead"]
)

total_portfolio_ead = float(
    baseline_components["total_ead"]
)

baseline_el_formula = (
    total_pd_ead * 0.40
)

moderate_el_formula = (
    total_pd_ead * 0.55
)

severe_el_formula = (
    total_pd_ead * 0.70
)


portfolio_el_rows = (
    portfolio_expected_loss
    .collect()
)

portfolio_el_lookup = {
    row["lgd_scenario"]: float(
        row["expected_loss_ngn"]
    )
    for row in portfolio_el_rows
}


el_reconciliation_checks = [
    (
        "Baseline EL reconciles to PD x EAD x 40%",
        bool(
            abs(
                portfolio_el_lookup["Baseline"] -
                baseline_el_formula
            ) < 0.01
        ),
        float(
            portfolio_el_lookup["Baseline"] -
            baseline_el_formula
        )
    ),
    (
        "Moderate LGD EL reconciles to PD x EAD x 55%",
        bool(
            abs(
                portfolio_el_lookup[
                    "Moderate LGD Sensitivity"
                ] -
                moderate_el_formula
            ) < 0.01
        ),
        float(
            portfolio_el_lookup[
                "Moderate LGD Sensitivity"
            ] -
            moderate_el_formula
        )
    ),
    (
        "Severe LGD EL reconciles to PD x EAD x 70%",
        bool(
            abs(
                portfolio_el_lookup[
                    "Severe LGD Sensitivity"
                ] -
                severe_el_formula
            ) < 0.01
        ),
        float(
            portfolio_el_lookup[
                "Severe LGD Sensitivity"
            ] -
            severe_el_formula
        )
    ),
    (
        "Baseline transaction EL sum reconciles",
        bool(
            abs(
                float(baseline_el_from_transaction_sum) -
                portfolio_el_lookup["Baseline"]
            ) < 0.01
        ),
        float(
            baseline_el_from_transaction_sum -
            portfolio_el_lookup["Baseline"]
        )
    )
]


el_reconciliation_gate = spark.createDataFrame(
    el_reconciliation_checks,
    schema=lgd_quality_schema
)

display(el_reconciliation_gate)


failed_el_reconciliation_checks = (
    el_reconciliation_gate
    .filter(
        F.col("passed") == F.lit(False)
    )
    .count()
)

if failed_el_reconciliation_checks > 0:
    raise ValueError(
        f"Expected-loss reconciliation failed: "
        f"{failed_el_reconciliation_checks} check(s) failed."
    )

print("Portfolio expected-loss reconciliation: PASS")


# -------------------------------------------------------------------
# Risk-band expected loss
# -------------------------------------------------------------------

expected_loss_risk_band = (
    expected_loss
    .groupBy(
        "lgd_scenario",
        "lgd",
        "risk_band",
        "risk_band_order"
    )
    .agg(
        F.countDistinct("transaction_id").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        F.sum(
            "pd_ead_exposure_ngn"
        ).alias(
            "pd_ead_exposure_ngn"
        ),
        F.sum(
            "expected_loss_ngn"
        ).alias(
            "expected_loss_ngn"
        ),
        F.sum("label").alias(
            "observed_default_count"
        ),
        F.sum(
            F.col("label") *
            F.col("ead_ngn")
        ).alias(
            "observed_default_exposure_ngn"
        )
    )
    .orderBy(
        "risk_band_order",
        "lgd"
    )
)


display(expected_loss_risk_band)


# -------------------------------------------------------------------
# Segment expected loss
# -------------------------------------------------------------------

portfolio_segment = (
    portfolio_risk
    .join(
        customer_kmeans_clusters
        .select(
            "customer_id",
            "cluster"
        ),
        on="customer_id",
        how="left"
    )
    .join(
        customer_segments
        .select(
            "cluster",
            "segment_label"
        ),
        on="cluster",
        how="left"
    )
)


segment_missing_count = (
    portfolio_segment
    .filter(
        F.col("segment_label").isNull()
    )
    .count()
)

if segment_missing_count > 0:
    raise ValueError(
        f"Segment assignment missing for "
        f"{segment_missing_count} transactions."
    )


expected_loss_segment = (
    portfolio_segment
    .crossJoin(
        F.broadcast(lgd_scenario_df)
    )
    .withColumn(
        "expected_loss_ngn",
        F.col("portfolio_pd") *
        F.col("ead_ngn") *
        F.col("lgd")
    )
    .withColumn(
        "pd_ead_exposure_ngn",
        F.col("portfolio_pd") *
        F.col("ead_ngn")
    )
    .groupBy(
        "lgd_scenario",
        "lgd",
        "cluster",
        "segment_label"
    )
    .agg(
        F.countDistinct("transaction_id").alias(
            "transaction_count"
        ),
        F.countDistinct("customer_id").alias(
            "customer_count"
        ),
        F.sum("ead_ngn").alias(
            "ead_ngn"
        ),
        F.avg("portfolio_pd").alias(
            "average_pd"
        ),
        F.sum(
            "pd_ead_exposure_ngn"
        ).alias(
            "pd_ead_exposure_ngn"
        ),
        F.sum(
            "expected_loss_ngn"
        ).alias(
            "expected_loss_ngn"
        ),
        F.sum("label").alias(
            "observed_default_count"
        )
    )
    .orderBy(
        "lgd",
        "cluster"
    )
)


display(expected_loss_segment)


# -------------------------------------------------------------------
# Save expected-loss transaction layer
# -------------------------------------------------------------------

expected_loss_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_expected_loss"
)

(
    expected_loss
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(expected_loss_output_path)
)

print(
    f"Expected-loss transaction layer saved to: "
    f"{expected_loss_output_path}"
)


# -------------------------------------------------------------------
# Save portfolio expected-loss summary
# -------------------------------------------------------------------

portfolio_expected_loss_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_expected_loss_portfolio_summary"
)

(
    portfolio_expected_loss
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        portfolio_expected_loss_output_path
    )
)

print(
    f"Portfolio expected-loss summary saved to: "
    f"{portfolio_expected_loss_output_path}"
)


# -------------------------------------------------------------------
# Save risk-band expected-loss summary
# -------------------------------------------------------------------

expected_loss_risk_band_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_expected_loss_risk_band"
)

(
    expected_loss_risk_band
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        expected_loss_risk_band_output_path
    )
)

print(
    f"Risk-band expected-loss summary saved to: "
    f"{expected_loss_risk_band_output_path}"
)


# -------------------------------------------------------------------
# Save segment expected-loss summary
# -------------------------------------------------------------------

expected_loss_segment_output_path = (
    f"{BNPL_RAW_PATH}/bnpl_expected_loss_segment"
)

(
    expected_loss_segment
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(
        expected_loss_segment_output_path
    )
)

print(
    f"Segment expected-loss summary saved to: "
    f"{expected_loss_segment_output_path}"
)


print("Expected-loss engine completed successfully.")

check,passed,observed_value
Exactly three LGD scenarios defined,true,3.0
LGD scenario names are unique,true,3.0
"All LGD values are within [0, 1]",true,0.0


LGD scenario quality gate: PASS


check,passed,observed_value
Expected-loss row count,true,1998738.0
Three scenario observations per transaction,true,1998738.0
Expected loss is non-negative and non-null,true,0.0
Transaction-level EL formula integrity,true,0.0


Transaction-level expected-loss integrity: PASS


lgd_scenario,lgd,lgd_description,scenario_transaction_rows,transaction_count,customer_count,total_ead_ngn,pd_ead_exposure_ngn,expected_loss_ngn,observed_default_count,observed_default_exposure_ngn,expected_loss_rate_of_ead
Baseline,0.4,Base severity assumption,666246,666246,421457,3.3317913042689316E10,3.289674922513504E9,1.3158699690054088E9,53310.0,3.8590464410232306E9,0.03949436950986159
Moderate LGD Sensitivity,0.55,Higher loss severity sensitivity,666246,666246,421457,3.331791304268933E10,3.2896749225134997E9,1.809321207382415E9,53310.0,3.8590464410232286E9,0.054304758076059
Severe LGD Sensitivity,0.7,Severe loss severity sensitivity,666246,666246,421457,3.33179130426893E10,3.289674922513504E9,2.302772445759457E9,53310.0,3.8590464410232277E9,0.06911514664225757


check,passed,observed_value
Baseline EL reconciles to PD x EAD x 40%,true,8.106231689453125E-6
Moderate LGD EL reconciles to PD x EAD x 55%,true,-1.0728836059570312E-5
Severe LGD EL reconciles to PD x EAD x 70%,true,6.67572021484375E-6
Baseline transaction EL sum reconciles,true,-2.6226043701171875E-6


Portfolio expected-loss reconciliation: PASS


lgd_scenario,lgd,risk_band,risk_band_order,transaction_count,customer_count,ead_ngn,average_pd,pd_ead_exposure_ngn,expected_loss_ngn,observed_default_count,observed_default_exposure_ngn
Baseline,0.4,Very Low,1,133214,121278,4.996942780677496E9,0.023637779717307714,1.1812810701237631E8,4.725124280495021E7,0.0,0.0
Moderate LGD Sensitivity,0.55,Very Low,1,133214,121278,4.9969427806775E9,0.023637779717307707,1.1812810701237625E8,6.497045885680644E7,0.0,0.0
Severe LGD Sensitivity,0.7,Very Low,1,133214,121278,4.996942780677503E9,0.023637779717307704,1.181281070123761E8,8.26896749086627E7,0.0,0.0
Baseline,0.4,Low,2,133296,120782,5.310990428719721E9,0.034710437086537,1.859639343202498E8,7.43855737281E7,0.0,0.0
Moderate LGD Sensitivity,0.55,Low,2,133296,120782,5.310990428719719E9,0.03471043708653701,1.8596393432024974E8,1.0228016387613738E8,0.0,0.0
Severe LGD Sensitivity,0.7,Low,2,133296,120782,5.310990428719717E9,0.03471043708653703,1.8596393432024968E8,1.3017475402417533E8,0.0,0.0
Baseline,0.4,Moderate,3,133206,119963,8.1601893833529215E9,0.05172307428834491,4.2584300628482383E8,1.7033720251393098E8,35.0,1322145.1858605407
Moderate LGD Sensitivity,0.55,Moderate,3,133206,119963,8.160189383352924E9,0.05172307428834494,4.258430062848236E8,2.3421365345665336E8,35.0,1322145.1858605407
Severe LGD Sensitivity,0.7,Moderate,3,133206,119963,8.160189383352915E9,0.051723074288344925,4.258430062848238E8,2.980901043993777E8,35.0,1322145.1858605407
Baseline,0.4,High,4,133290,120632,6.462766710921423E9,0.07877612773939319,5.1580579967408955E8,2.0632231986963344E8,1476.0,7.267217819862375E7


lgd_scenario,lgd,cluster,segment_label,transaction_count,customer_count,ead_ngn,average_pd,pd_ead_exposure_ngn,expected_loss_ngn,observed_default_count
Baseline,0.4,0,Low-Engagement / Low-Exposure,57678,48860,2.441684880401838E9,0.07550229633510433,2.254896119665721E8,9.019584478662887E7,4182.0
Baseline,0.4,1,Active / Higher-Exposure,608568,372597,3.087622816228848E10,0.08044323485951259,3.064185310546967E9,1.225674124218787E9,49128.0
Moderate LGD Sensitivity,0.55,0,Low-Engagement / Low-Exposure,57678,48860,2.4416848804018373E9,0.07550229633510433,2.254896119665722E8,1.2401928658161475E8,4182.0
Moderate LGD Sensitivity,0.55,1,Active / Higher-Exposure,608568,372597,3.087622816228845E10,0.08044323485951257,3.0641853105469675E9,1.6853019208008358E9,49128.0
Severe LGD Sensitivity,0.7,0,Low-Engagement / Low-Exposure,57678,48860,2.441684880401838E9,0.07550229633510434,2.2548961196657223E8,1.5784272837660056E8,4182.0
Severe LGD Sensitivity,0.7,1,Active / Higher-Exposure,608568,372597,3.0876228162288445E10,0.08044323485951255,3.0641853105469694E9,2.144929717382875E9,49128.0


Expected-loss transaction layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_expected_loss
Portfolio expected-loss summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_portfolio_summary
Risk-band expected-loss summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_risk_band
Segment expected-loss summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_segment
Expected-loss engine completed successfully.


## Historical Macro Context and Stress Testing

Historical Nigerian macroeconomic conditions are incorporated as stress context rather than as a causal macro-default model.

The 2022–2024 macro panel contains 36 monthly observations covering:

- Inflation year-on-year
- Inflation month-on-month
- NGN/USD exchange rate
- Prime lending rate
- Maximum lending rate
- Monetary Policy Rate

A percentile-based composite macro stress index is constructed to identify periods of relatively elevated adverse macro conditions.

The 75th percentile is used as a moderate historical stress threshold and the 90th percentile as a severe historical stress threshold. August 2024 and November 2024 provide historical context anchors for the moderate and severe scenarios respectively.

The resulting PD multipliers are management sensitivity assumptions, not estimated causal effects of macroeconomic variables on BNPL default.

In [0]:
# ================================================================
# Historical Macro Context and Reproducible Stress Scenario Framework
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from datetime import date

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

MACRO_PATH = "/Volumes/workspace/default/macro_raw/Nigeria_Macro_Final_2022_2024.csv"
MACRO_OUTPUT_PATH = "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_context"

# ----------------------------------------------------------------
# Load macro dataset
# ----------------------------------------------------------------

macro_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(MACRO_PATH)
)

print("Macro source columns:")
print(macro_raw.columns)

# ----------------------------------------------------------------
# Normalize column names
# ----------------------------------------------------------------

macro = macro_raw

for original_col in macro_raw.columns:
    normalized_col = (
        original_col.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "pct")
        .replace(".", "_")
    )

    if original_col != normalized_col:
        macro = macro.withColumnRenamed(
            original_col,
            normalized_col
        )

print("\nNormalized macro columns:")
print(macro.columns)

# ----------------------------------------------------------------
# Validate the actual expected schema
# ----------------------------------------------------------------

required_macro_columns = {
    "month",
    "inflation_mom",
    "inflation_yoy",
    "avg_ngn_usd_ie",
    "prime_lending_rate",
    "max_lending_rate",
    "mpr_pct"
}

missing_macro_columns = (
    required_macro_columns
    - set(macro.columns)
)

if missing_macro_columns:
    raise ValueError(
        f"Required macro columns are missing: "
        f"{sorted(missing_macro_columns)}"
    )

# ----------------------------------------------------------------
# Build controlled macro layer
#
# avg_ngn_usd_ie is used as the FX indicator.
# mpr_pct is used as the Monetary Policy Rate.
# ----------------------------------------------------------------

macro_controlled = (
    macro
    .select(
        F.to_date(F.col("month")).alias("macro_date"),

        F.col("inflation_yoy")
        .cast("double")
        .alias("inflation_yoy"),

        F.col("inflation_mom")
        .cast("double")
        .alias("inflation_mom"),

        F.col("avg_ngn_usd_ie")
        .cast("double")
        .alias("fx_rate"),

        F.col("prime_lending_rate")
        .cast("double")
        .alias("prime_lending_rate"),

        F.col("max_lending_rate")
        .cast("double")
        .alias("max_lending_rate"),

        F.col("mpr_pct")
        .cast("double")
        .alias("mpr")
    )
    .filter(F.col("macro_date").isNotNull())
    .dropDuplicates(["macro_date"])
    .orderBy("macro_date")
)

# ----------------------------------------------------------------
# Audit macro observations
# ----------------------------------------------------------------

indicator_columns = [
    "inflation_yoy",
    "inflation_mom",
    "fx_rate",
    "prime_lending_rate",
    "max_lending_rate",
    "mpr"
]

macro_audit = (
    macro_controlled
    .agg(
        F.count("*").alias("observation_count"),
        F.min("macro_date").alias("min_date"),
        F.max("macro_date").alias("max_date"),
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(f"{c}_nulls")
            for c in indicator_columns
        ]
    )
)

print("\nMacro source audit:")
display(macro_audit)

# ----------------------------------------------------------------
# Validate expected 36 monthly observations
# ----------------------------------------------------------------

macro_observation_count = macro_controlled.count()

if macro_observation_count != 36:
    raise ValueError(
        f"Expected 36 monthly macro observations for 2022-2024, "
        f"but found {macro_observation_count}."
    )

# ----------------------------------------------------------------
# Validate no missing indicator values
# ----------------------------------------------------------------

macro_missing_count = (
    macro_controlled
    .filter(
        reduce(
            lambda a, b: a | b,
            [
                F.col(c).isNull()
                for c in indicator_columns
            ]
        )
    )
    .count()
)

if macro_missing_count > 0:
    raise ValueError(
        f"Macro indicator completeness check failed. "
        f"{macro_missing_count} rows contain missing indicator values."
    )

# ----------------------------------------------------------------
# Percentile-based adverse stress scores
#
# Higher percentile = greater historical adverse pressure.
#
# These scores provide historical severity context only.
# They are NOT causal estimates of BNPL default sensitivity.
# ----------------------------------------------------------------

macro_ranked = macro_controlled

for indicator in indicator_columns:

    macro_ranked = macro_ranked.withColumn(
        f"{indicator}_stress_pct",
        F.percent_rank().over(
            Window.orderBy(F.col(indicator))
        )
    )

# ----------------------------------------------------------------
# Composite macro stress index
# ----------------------------------------------------------------

stress_components = [
    F.col(f"{indicator}_stress_pct")
    for indicator in indicator_columns
]

macro_ranked = (
    macro_ranked
    .withColumn(
        "macro_stress_index",
        sum(stress_components) / F.lit(
            float(len(stress_components))
        )
    )
    .withColumn(
        "macro_stress_index",
        F.round(F.col("macro_stress_index"), 6)
    )
)

# ----------------------------------------------------------------
# Historical severity thresholds
#
# 75th percentile = moderate historical stress context
# 90th percentile = severe historical stress context
# ----------------------------------------------------------------

stress_threshold_row = (
    macro_ranked
    .agg(
        F.percentile_approx(
            "macro_stress_index",
            0.75,
            10000
        )
        .cast("double")
        .alias("moderate_threshold"),

        F.percentile_approx(
            "macro_stress_index",
            0.90,
            10000
        )
        .cast("double")
        .alias("severe_threshold")
    )
    .collect()[0]
)

moderate_threshold = float(
    stress_threshold_row["moderate_threshold"]
)

severe_threshold = float(
    stress_threshold_row["severe_threshold"]
)

# ----------------------------------------------------------------
# Historical stress classification
# ----------------------------------------------------------------

macro_ranked = (
    macro_ranked
    .withColumn(
        "historical_stress_class",
        F.when(
            F.col("macro_stress_index")
            >= F.lit(severe_threshold),
            F.lit("Severe historical stress")
        )
        .when(
            F.col("macro_stress_index")
            >= F.lit(moderate_threshold),
            F.lit("Moderate historical stress")
        )
        .otherwise(
            F.lit("Normal / lower-stress")
        )
    )
)

# ----------------------------------------------------------------
# Historical scenario anchors
#
# August 2024 = moderate macro context
# November 2024 = severe macro context
#
# These anchors provide historical context only.
# ----------------------------------------------------------------

moderate_anchor_date = date(2024, 8, 1)
severe_anchor_date = date(2024, 11, 1)

anchor_rows = (
    macro_ranked
    .filter(
        F.col("macro_date").isin(
            moderate_anchor_date,
            severe_anchor_date
        )
    )
    .withColumn(
        "scenario_anchor",
        F.when(
            F.col("macro_date") == F.lit(moderate_anchor_date),
            F.lit("Moderate Macro Stress Anchor")
        )
        .when(
            F.col("macro_date") == F.lit(severe_anchor_date),
            F.lit("Severe Macro Stress Anchor")
        )
    )
    .select(
        "macro_date",
        "scenario_anchor",
        "macro_stress_index",
        "historical_stress_class",
        *indicator_columns
    )
    .orderBy("macro_date")
)

anchor_count = anchor_rows.count()

if anchor_count != 2:
    raise ValueError(
        f"Expected exactly two macro anchor observations, "
        f"but found {anchor_count}."
    )

# ----------------------------------------------------------------
# Reproducible stress scenario assumptions
#
# IMPORTANT:
# The PD multipliers are management sensitivity assumptions.
# They are NOT empirically estimated causal coefficients.
# ----------------------------------------------------------------

scenario_schema = T.StructType([
    T.StructField("scenario", T.StringType(), False),
    T.StructField("scenario_type", T.StringType(), False),
    T.StructField("macro_anchor_date", T.DateType(), True),
    T.StructField("pd_multiplier", T.DoubleType(), False),
    T.StructField("lgd", T.DoubleType(), False),
    T.StructField("methodology_note", T.StringType(), False)
])

scenario_rows = [
    (
        "Baseline",
        "Baseline",
        None,
        1.00,
        0.40,
        "Observed portfolio-normalized PD with baseline 40% LGD; no macro stress multiplier."
    ),
    (
        "Moderate Macro Stress",
        "Macro stress",
        moderate_anchor_date,
        1.25,
        0.55,
        "August 2024 provides historical severity context. A 1.25x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."
    ),
    (
        "Severe Macro Stress",
        "Macro stress",
        severe_anchor_date,
        1.50,
        0.70,
        "November 2024 provides historical severity context. A 1.50x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."
    )
]

scenario_df = spark.createDataFrame(
    scenario_rows,
    schema=scenario_schema
)

# ----------------------------------------------------------------
# Scenario quality gate
# ----------------------------------------------------------------

scenario_quality_schema = T.StructType([
    T.StructField("check", T.StringType(), False),
    T.StructField("passed", T.BooleanType(), False),
    T.StructField("observed_value", T.DoubleType(), False)
])

scenario_count = scenario_df.count()

unique_scenario_count = (
    scenario_df
    .select("scenario")
    .distinct()
    .count()
)

invalid_pd_multiplier_count = (
    scenario_df
    .filter(
        (F.col("pd_multiplier") < 1.0) |
        (F.col("pd_multiplier") > 2.0)
    )
    .count()
)

invalid_lgd_count = (
    scenario_df
    .filter(
        (F.col("lgd") < 0.0) |
        (F.col("lgd") > 1.0)
    )
    .count()
)

scenario_quality_rows = [
    (
        "Exactly three stress scenarios defined",
        scenario_count == 3,
        float(scenario_count)
    ),
    (
        "Scenario names are unique",
        unique_scenario_count == 3,
        float(unique_scenario_count)
    ),
    (
        "PD multipliers are within valid stress range",
        invalid_pd_multiplier_count == 0,
        float(invalid_pd_multiplier_count)
    ),
    (
        "LGD values are within [0, 1]",
        invalid_lgd_count == 0,
        float(invalid_lgd_count)
    ),
    (
        "Both historical macro anchors are present",
        anchor_count == 2,
        float(anchor_count)
    )
]

scenario_quality = spark.createDataFrame(
    scenario_quality_rows,
    schema=scenario_quality_schema
)

display(scenario_quality)

if scenario_quality.filter(
    ~F.col("passed")
).count() > 0:
    raise ValueError(
        "Macro stress scenario quality gate FAILED."
    )

print("Macro stress scenario quality gate: PASS")

# ----------------------------------------------------------------
# Save historical macro context layer
# ----------------------------------------------------------------

(
    macro_ranked
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(MACRO_OUTPUT_PATH)
)

print(
    f"\nMacro stress context saved to: "
    f"{MACRO_OUTPUT_PATH}"
)

# ----------------------------------------------------------------
# Review outputs
# ----------------------------------------------------------------

print("\nHistorical macro stress anchors:")
display(anchor_rows)

print("\nStress scenario assumptions:")

display(
    scenario_df
    .orderBy(
        F.when(
            F.col("scenario") == "Baseline", 1
        )
        .when(
            F.col("scenario") == "Moderate Macro Stress", 2
        )
        .when(
            F.col("scenario") == "Severe Macro Stress", 3
        )
        .otherwise(99)
    )
)

print(
    f"\nHistorical moderate stress threshold: "
    f"{moderate_threshold:.6f}"
)

print(
    f"Historical severe stress threshold: "
    f"{severe_threshold:.6f}"
)

print(
    f"\nMacro observations validated: "
    f"{macro_observation_count}"
)

Macro source columns:
['month', 'cpi_all_items', 'inflation_mom', 'inflation_yoy', 'avg_ngn_usd_ie', 'prime_lending_rate', 'max_lending_rate', 'mpr_pct']

Normalized macro columns:
['month', 'cpi_all_items', 'inflation_mom', 'inflation_yoy', 'avg_ngn_usd_ie', 'prime_lending_rate', 'max_lending_rate', 'mpr_pct']

Macro source audit:


observation_count,min_date,max_date,inflation_yoy_nulls,inflation_mom_nulls,fx_rate_nulls,prime_lending_rate_nulls,max_lending_rate_nulls,mpr_nulls
36,2022-01-01,2024-12-01,0,0,0,0,0,0


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


check,passed,observed_value
Exactly three stress scenarios defined,true,3.0
Scenario names are unique,true,3.0
PD multipliers are within valid stress range,true,0.0
"LGD values are within [0, 1]",true,0.0
Both historical macro anchors are present,true,2.0


Macro stress scenario quality gate: PASS

Macro stress context saved to: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_context

Historical macro stress anchors:


macro_date,scenario_anchor,macro_stress_index,historical_stress_class,inflation_yoy,inflation_mom,fx_rate,prime_lending_rate,max_lending_rate,mpr
2024-08-01,Moderate Macro Stress Anchor,0.82381,Moderate historical stress,32.15,2.22,1587.39,17.01,29.93,26.75
2024-11-01,Severe Macro Stress Anchor,0.957143,Severe historical stress,34.6,2.64,1670.78,18.39,31.06,27.5



Stress scenario assumptions:


scenario,scenario_type,macro_anchor_date,pd_multiplier,lgd,methodology_note
Baseline,Baseline,null,1.0,0.4,Observed portfolio-normalized PD with baseline 40% LGD; no macro stress multiplier.
Moderate Macro Stress,Macro stress,2024-08-01,1.25,0.55,"August 2024 provides historical severity context. A 1.25x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."
Severe Macro Stress,Macro stress,2024-11-01,1.5,0.7,"November 2024 provides historical severity context. A 1.50x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."



Historical moderate stress threshold: 0.733333
Historical severe stress threshold: 0.871429

Macro observations validated: 36


## Macro Stress Scenarios

Three portfolio scenarios are evaluated:

| Scenario | PD Multiplier | LGD |
|---|---:|---:|
| Baseline | 1.00× | 40% |
| Moderate Macro Stress | 1.25× | 55% |
| Severe Macro Stress | 1.50× | 70% |

For each scenario:

\[
PD_{stress} = min(PD_{portfolio} * PD\ multiplier, 1)
\]

\[
EL_{stress} = PD_{stress} * EAD * LGD
\]

EAD is held constant because the analysis does not estimate behavioural drawdown or portfolio growth under stress.

The scenarios should therefore be interpreted as reproducible management sensitivity cases, not probabilistic forecasts of future portfolio losses.

In [0]:
# ================================================================
# Portfolio Macro Stress Engine
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

STRESS_CONTEXT_PATH = "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_context"

STRESS_TRANSACTION_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_transactions"
)

STRESS_PORTFOLIO_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_portfolio_summary"
)

STRESS_RISK_BAND_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_risk_band"
)

STRESS_SEGMENT_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_segment"
)

# ----------------------------------------------------------------
# Load authoritative frozen analytical layers
# ----------------------------------------------------------------

portfolio_risk_stress = (
    spark.read
    .format("delta")
    .load("/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk")
)

risk_band_stress = (
    spark.read
    .format("delta")
    .load("/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_bands")
)

customer_clusters_stress = (
    spark.read
    .format("delta")
    .load("/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters")
)

customer_segments_stress = (
    spark.read
    .format("delta")
    .load("/Volumes/workspace/default/bnpl_raw/final_bnpl_customer_segments")
)

# ----------------------------------------------------------------
# Reconstruct the scenario table explicitly
#
# This keeps the stress engine self-contained and avoids relying
# on temporary notebook state.
# ----------------------------------------------------------------

scenario_schema = T.StructType([
    T.StructField("scenario", T.StringType(), False),
    T.StructField("scenario_type", T.StringType(), False),
    T.StructField("macro_anchor_date", T.DateType(), True),
    T.StructField("pd_multiplier", T.DoubleType(), False),
    T.StructField("lgd", T.DoubleType(), False),
    T.StructField("methodology_note", T.StringType(), False)
])

scenario_rows = [
    (
        "Baseline",
        "Baseline",
        None,
        1.00,
        0.40,
        "Observed portfolio-normalized PD with baseline 40% LGD; no macro stress multiplier."
    ),
    (
        "Moderate Macro Stress",
        "Macro stress",
        date(2024, 8, 1),
        1.25,
        0.55,
        "August 2024 provides historical severity context. A 1.25x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."
    ),
    (
        "Severe Macro Stress",
        "Macro stress",
        date(2024, 11, 1),
        1.50,
        0.70,
        "November 2024 provides historical severity context. A 1.50x PD multiplier is a transparent management sensitivity assumption, not a causal macro-default estimate."
    )
]

scenario_df = spark.createDataFrame(
    scenario_rows,
    schema=scenario_schema
)

# ----------------------------------------------------------------
# Validate required portfolio fields
# ----------------------------------------------------------------

required_portfolio_columns = {
    "transaction_id",
    "customer_id",
    "portfolio_pd",
    "ead_ngn",
    "pd_ead_loss_exposure_ngn"
}

missing_portfolio_columns = (
    required_portfolio_columns
    - set(portfolio_risk_stress.columns)
)

if missing_portfolio_columns:
    raise ValueError(
        f"Required portfolio-risk columns are missing: "
        f"{sorted(missing_portfolio_columns)}"
    )

# ----------------------------------------------------------------
# Baseline population
# ----------------------------------------------------------------

portfolio_risk_stress = (
    portfolio_risk_stress
    .select(
        "transaction_id",
        "customer_id",
        "purchase_date",
        "principal_ngn",
        "portfolio_pd",
        "ead_ngn",
        "pd_ead_loss_exposure_ngn"
    )
)

baseline_transaction_count = portfolio_risk_stress.count()

baseline_customer_count = (
    portfolio_risk_stress
    .select("customer_id")
    .distinct()
    .count()
)

if baseline_transaction_count != 666246:
    raise ValueError(
        f"Unexpected portfolio population. "
        f"Expected 666246 transactions, found "
        f"{baseline_transaction_count}."
    )

# ----------------------------------------------------------------
# Generate transaction-level stress scenarios
# ----------------------------------------------------------------

stress_transactions = (
    portfolio_risk_stress
    .crossJoin(F.broadcast(scenario_df))
    .withColumn(
        "stressed_pd",
        F.least(
            F.col("portfolio_pd") * F.col("pd_multiplier"),
            F.lit(1.0)
        )
    )
    .withColumn(
        "stressed_ead_ngn",
        F.col("ead_ngn")
    )
    .withColumn(
        "stressed_lgd",
        F.col("lgd")
    )
    .withColumn(
        "stressed_pd_ead_exposure_ngn",
        F.col("stressed_pd") *
        F.col("stressed_ead_ngn")
    )
    .withColumn(
        "stressed_expected_loss_ngn",
        F.col("stressed_pd") *
        F.col("stressed_ead_ngn") *
        F.col("stressed_lgd")
    )
)

# ----------------------------------------------------------------
# Baseline expected loss for incremental comparison
# ----------------------------------------------------------------

baseline_expected_loss = (
    portfolio_risk_stress
    .agg(
        F.sum(
            F.col("portfolio_pd") *
            F.col("ead_ngn") *
            F.lit(0.40)
        ).alias("baseline_expected_loss_ngn")
    )
    .collect()[0]["baseline_expected_loss_ngn"]
)

baseline_expected_loss = float(baseline_expected_loss)

stress_transactions = (
    stress_transactions
    .withColumn(
        "baseline_expected_loss_ngn",
        F.lit(baseline_expected_loss)
    )
)

# ----------------------------------------------------------------
# Transaction-level stress quality checks
# ----------------------------------------------------------------

expected_stress_rows = baseline_transaction_count * 3

actual_stress_rows = stress_transactions.count()

invalid_stressed_pd = (
    stress_transactions
    .filter(
        F.col("stressed_pd").isNull() |
        (F.col("stressed_pd") < 0) |
        (F.col("stressed_pd") > 1)
    )
    .count()
)

invalid_stressed_ead = (
    stress_transactions
    .filter(
        F.col("stressed_ead_ngn").isNull() |
        (F.col("stressed_ead_ngn") <= 0)
    )
    .count()
)

invalid_stressed_lgd = (
    stress_transactions
    .filter(
        F.col("stressed_lgd").isNull() |
        (F.col("stressed_lgd") < 0) |
        (F.col("stressed_lgd") > 1)
    )
    .count()
)

formula_mismatch = (
    stress_transactions
    .filter(
        F.abs(
            F.col("stressed_expected_loss_ngn") -
            (
                F.col("stressed_pd") *
                F.col("stressed_ead_ngn") *
                F.col("stressed_lgd")
            )
        ) > F.lit(0.01)
    )
    .count()
)

stress_quality_schema = T.StructType([
    T.StructField("check", T.StringType(), False),
    T.StructField("passed", T.BooleanType(), False),
    T.StructField("observed_value", T.DoubleType(), False)
])

stress_quality_rows = [
    (
        "Expected transaction-scenario row count",
        actual_stress_rows == expected_stress_rows,
        float(actual_stress_rows)
    ),
    (
        "Stressed PD values are within [0, 1]",
        invalid_stressed_pd == 0,
        float(invalid_stressed_pd)
    ),
    (
        "Stressed EAD values are positive and non-null",
        invalid_stressed_ead == 0,
        float(invalid_stressed_ead)
    ),
    (
        "Stressed LGD values are within [0, 1]",
        invalid_stressed_lgd == 0,
        float(invalid_stressed_lgd)
    ),
    (
        "Stressed expected-loss formula integrity",
        formula_mismatch == 0,
        float(formula_mismatch)
    )
]

stress_quality = spark.createDataFrame(
    stress_quality_rows,
    schema=stress_quality_schema
)

display(stress_quality)

if stress_quality.filter(
    ~F.col("passed")
).count() > 0:
    raise ValueError(
        "Transaction-level macro stress quality gate FAILED."
    )

print("Transaction-level macro stress quality gate: PASS")

# ----------------------------------------------------------------
# Portfolio-level stress aggregation
# ----------------------------------------------------------------

stress_portfolio = (
    stress_transactions
    .groupBy(
        "scenario",
        "scenario_type",
        "macro_anchor_date",
        "pd_multiplier",
        "stressed_lgd"
    )
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct(
            "customer_id"
        ).alias("customer_count"),

        F.sum(
            "stressed_ead_ngn"
        ).alias("stressed_ead_ngn"),

        F.avg(
            "stressed_pd"
        ).alias("transaction_weighted_stressed_pd"),

        (
            F.sum(
                F.col("stressed_pd") *
                F.col("stressed_ead_ngn")
            )
            /
            F.sum("stressed_ead_ngn")
        ).alias("exposure_weighted_stressed_pd"),

        F.sum(
            "stressed_pd_ead_exposure_ngn"
        ).alias("stressed_pd_ead_exposure_ngn"),

        F.sum(
            "stressed_expected_loss_ngn"
        ).alias("stressed_expected_loss_ngn")
    )
    .withColumn(
        "expected_loss_rate_of_ead",
        F.col("stressed_expected_loss_ngn") /
        F.col("stressed_ead_ngn")
    )
)

# ----------------------------------------------------------------
# Add baseline and incremental loss
# ----------------------------------------------------------------

stress_portfolio = (
    stress_portfolio
    .withColumn(
        "baseline_expected_loss_ngn",
        F.lit(baseline_expected_loss)
    )
    .withColumn(
        "incremental_expected_loss_ngn",
        F.col("stressed_expected_loss_ngn") -
        F.col("baseline_expected_loss_ngn")
    )
    .withColumn(
        "stress_expected_loss_multiple",
        F.col("stressed_expected_loss_ngn") /
        F.col("baseline_expected_loss_ngn")
    )
    .withColumn(
        "stress_expected_loss_increase_pct",
        (
            F.col("stressed_expected_loss_ngn") /
            F.col("baseline_expected_loss_ngn") -
            F.lit(1.0)
        ) * F.lit(100.0)
    )
)

# ----------------------------------------------------------------
# Portfolio reconciliation
# ----------------------------------------------------------------

portfolio_reconciliation = (
    stress_portfolio
    .select(
        "scenario",
        "transaction_count",
        "customer_count",
        "stressed_ead_ngn",
        "stressed_pd_ead_exposure_ngn",
        "stressed_expected_loss_ngn",
        "incremental_expected_loss_ngn",
        "stress_expected_loss_multiple",
        "stress_expected_loss_increase_pct"
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
        .when(F.col("scenario") == "Moderate Macro Stress", 2)
        .when(F.col("scenario") == "Severe Macro Stress", 3)
        .otherwise(99)
    )
)

display(portfolio_reconciliation)

# ----------------------------------------------------------------
# Validate baseline against the previously frozen EL output
# ----------------------------------------------------------------

baseline_stress_value = (
    stress_portfolio
    .filter(F.col("scenario") == "Baseline")
    .select("stressed_expected_loss_ngn")
    .collect()[0][0]
)

baseline_difference = (
    float(baseline_stress_value) -
    baseline_expected_loss
)

if abs(baseline_difference) > 0.10:
    raise ValueError(
        f"Baseline stress reconciliation failed. "
        f"Difference = {baseline_difference}"
    )

# ----------------------------------------------------------------
# Save transaction-level stress output
# ----------------------------------------------------------------

(
    stress_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(STRESS_TRANSACTION_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Save portfolio-level stress output
# ----------------------------------------------------------------

(
    stress_portfolio
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(STRESS_PORTFOLIO_OUTPUT_PATH)
)

print(
    f"\nTransaction stress layer saved to: "
    f"{STRESS_TRANSACTION_OUTPUT_PATH}"
)

print(
    f"Portfolio stress summary saved to: "
    f"{STRESS_PORTFOLIO_OUTPUT_PATH}"
)

print(
    "\nMacro stress engine completed successfully."
)

check,passed,observed_value
Expected transaction-scenario row count,true,1998738.0
"Stressed PD values are within [0, 1]",true,0.0
Stressed EAD values are positive and non-null,true,0.0
"Stressed LGD values are within [0, 1]",true,0.0
Stressed expected-loss formula integrity,true,0.0


Transaction-level macro stress quality gate: PASS


scenario,transaction_count,customer_count,stressed_ead_ngn,stressed_pd_ead_exposure_ngn,stressed_expected_loss_ngn,incremental_expected_loss_ngn,stress_expected_loss_multiple,stress_expected_loss_increase_pct
Baseline,666246,421457,3.3317913042690228E10,3.28967492251355E9,1.315869969005415E9,1.5020370483398438E-5,1.0000000000000113,1.1324274851176597E-12
Moderate Macro Stress,666246,421457,3.3317913042690308E10,4.112093653141924E9,2.2616515092280593E9,9.457815402226593E8,1.7187500000000213,71.87500000000213
Severe Macro Stress,666246,421457,3.3317913042690308E10,4.93451238377032E9,3.4541586686392164E9,2.1382886996338165E9,2.6250000000000315,162.50000000000315



Transaction stress layer saved to: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_transactions
Portfolio stress summary saved to: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_portfolio_summary

Macro stress engine completed successfully.


In [0]:
# ================================================================
# Stress Impact by Risk Band and Customer Segment
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

STRESS_TRANSACTION_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_transactions"
)

STRESS_PORTFOLIO_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_portfolio_summary"
)

STRESS_RISK_BAND_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_risk_band"
)

STRESS_SEGMENT_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_segment"
)

RISK_BAND_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_bands"
)

CUSTOMER_CLUSTERS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/customer_kmeans_clusters"
)

CUSTOMER_SEGMENTS_PATH = (
    "/Volumes/workspace/default/bnpl_raw/final_bnpl_customer_segments"
)

# ----------------------------------------------------------------
# Load stress transactions
# ----------------------------------------------------------------

stress_tx = (
    spark.read
    .format("delta")
    .load(STRESS_TRANSACTION_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Load authoritative risk-band assignment
# ----------------------------------------------------------------

risk_bands = (
    spark.read
    .format("delta")
    .load(RISK_BAND_PATH)
    .select(
        "transaction_id",
        "risk_decile",
        "risk_band",
        "risk_band_order"
    )
    .dropDuplicates(["transaction_id"])
)

# ----------------------------------------------------------------
# Validate risk-band uniqueness
# ----------------------------------------------------------------

risk_band_duplicate_count = (
    risk_bands
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if risk_band_duplicate_count > 0:
    raise ValueError(
        f"Risk-band assignment is not unique. "
        f"Duplicate transaction IDs: {risk_band_duplicate_count}"
    )

# ----------------------------------------------------------------
# Attach risk bands to stress transactions
# ----------------------------------------------------------------

stress_band = (
    stress_tx
    .join(
        risk_bands,
        on="transaction_id",
        how="left"
    )
)

missing_risk_band_count = (
    stress_band
    .filter(F.col("risk_band").isNull())
    .count()
)

if missing_risk_band_count > 0:
    raise ValueError(
        f"{missing_risk_band_count} stress rows are missing risk-band assignments."
    )

# ----------------------------------------------------------------
# Risk-band stress aggregation
# ----------------------------------------------------------------

stress_risk_band = (
    stress_band
    .groupBy(
        "scenario",
        "pd_multiplier",
        "stressed_lgd",
        "risk_band",
        "risk_band_order"
    )
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct(
            "customer_id"
        ).alias("customer_count"),

        F.sum(
            "stressed_ead_ngn"
        ).alias("ead_ngn"),

        F.avg(
            "stressed_pd"
        ).alias("average_stressed_pd"),

        (
            F.sum(
                F.col("stressed_pd") *
                F.col("stressed_ead_ngn")
            ) /
            F.sum("stressed_ead_ngn")
        ).alias("exposure_weighted_stressed_pd"),

        F.sum(
            "stressed_pd_ead_exposure_ngn"
        ).alias("stressed_pd_ead_exposure_ngn"),

        F.sum(
            "stressed_expected_loss_ngn"
        ).alias("stressed_expected_loss_ngn")
    )
)

# ----------------------------------------------------------------
# Calculate portfolio shares
# ----------------------------------------------------------------

stress_risk_band_totals = (
    stress_risk_band
    .groupBy("scenario")
    .agg(
        F.sum("ead_ngn").alias("portfolio_ead_ngn"),
        F.sum("stressed_expected_loss_ngn")
        .alias("portfolio_expected_loss_ngn")
    )
)

stress_risk_band = (
    stress_risk_band
    .join(
        stress_risk_band_totals,
        on="scenario",
        how="left"
    )
    .withColumn(
        "ead_share_pct",
        F.col("ead_ngn") /
        F.col("portfolio_ead_ngn") *
        F.lit(100.0)
    )
    .withColumn(
        "expected_loss_share_pct",
        F.col("stressed_expected_loss_ngn") /
        F.col("portfolio_expected_loss_ngn") *
        F.lit(100.0)
    )
    .withColumn(
        "expected_loss_rate_of_ead",
        F.col("stressed_expected_loss_ngn") /
        F.col("ead_ngn")
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
        .when(F.col("scenario") == "Moderate Macro Stress", 2)
        .when(F.col("scenario") == "Severe Macro Stress", 3)
        .otherwise(99),
        F.col("risk_band_order")
    )
)

# ----------------------------------------------------------------
# Load customer segmentation
# ----------------------------------------------------------------

customer_clusters = (
    spark.read
    .format("delta")
    .load(CUSTOMER_CLUSTERS_PATH)
)

customer_segments = (
    spark.read
    .format("delta")
    .load(CUSTOMER_SEGMENTS_PATH)
)

# ----------------------------------------------------------------
# Identify required segment columns
# ----------------------------------------------------------------

required_cluster_columns = {
    "customer_id",
    "cluster"
}

required_segment_columns = {
    "cluster",
    "segment_label"
}

missing_cluster_columns = (
    required_cluster_columns -
    set(customer_clusters.columns)
)

missing_segment_columns = (
    required_segment_columns -
    set(customer_segments.columns)
)

if missing_cluster_columns:
    raise ValueError(
        f"Missing customer-cluster columns: "
        f"{sorted(missing_cluster_columns)}"
    )

if missing_segment_columns:
    raise ValueError(
        f"Missing customer-segment columns: "
        f"{sorted(missing_segment_columns)}"
    )

# ----------------------------------------------------------------
# Build controlled customer segment mapping
# ----------------------------------------------------------------

customer_segment_map = (
    customer_clusters
    .select(
        "customer_id",
        "cluster"
    )
    .dropDuplicates(["customer_id"])
    .join(
        customer_segments
        .select(
            "cluster",
            "segment_label"
        )
        .dropDuplicates(["cluster"]),
        on="cluster",
        how="left"
    )
)

missing_segment_count = (
    customer_segment_map
    .filter(F.col("segment_label").isNull())
    .count()
)

if missing_segment_count > 0:
    raise ValueError(
        f"{missing_segment_count} customers have no segment label."
    )

# ----------------------------------------------------------------
# Attach segment labels to stress transactions
# ----------------------------------------------------------------

stress_segment = (
    stress_tx
    .join(
        customer_segment_map,
        on="customer_id",
        how="left"
    )
)

missing_stress_segment_count = (
    stress_segment
    .filter(F.col("segment_label").isNull())
    .count()
)

if missing_stress_segment_count > 0:
    raise ValueError(
        f"{missing_stress_segment_count} stress rows are missing "
        f"customer-segment assignments."
    )

# ----------------------------------------------------------------
# Segment stress aggregation
# ----------------------------------------------------------------

stress_segment_summary = (
    stress_segment
    .groupBy(
        "scenario",
        "pd_multiplier",
        "stressed_lgd",
        "cluster",
        "segment_label"
    )
    .agg(
        F.count("*").alias("transaction_count"),

        F.countDistinct(
            "customer_id"
        ).alias("customer_count"),

        F.sum(
            "stressed_ead_ngn"
        ).alias("ead_ngn"),

        F.avg(
            "stressed_pd"
        ).alias("average_stressed_pd"),

        (
            F.sum(
                F.col("stressed_pd") *
                F.col("stressed_ead_ngn")
            ) /
            F.sum("stressed_ead_ngn")
        ).alias("exposure_weighted_stressed_pd"),

        F.sum(
            "stressed_pd_ead_exposure_ngn"
        ).alias("stressed_pd_ead_exposure_ngn"),

        F.sum(
            "stressed_expected_loss_ngn"
        ).alias("stressed_expected_loss_ngn")
    )
)

# ----------------------------------------------------------------
# Calculate segment portfolio shares
# ----------------------------------------------------------------

stress_segment_totals = (
    stress_segment_summary
    .groupBy("scenario")
    .agg(
        F.sum("ead_ngn").alias("portfolio_ead_ngn"),
        F.sum("stressed_expected_loss_ngn")
        .alias("portfolio_expected_loss_ngn")
    )
)

stress_segment_summary = (
    stress_segment_summary
    .join(
        stress_segment_totals,
        on="scenario",
        how="left"
    )
    .withColumn(
        "ead_share_pct",
        F.col("ead_ngn") /
        F.col("portfolio_ead_ngn") *
        F.lit(100.0)
    )
    .withColumn(
        "expected_loss_share_pct",
        F.col("stressed_expected_loss_ngn") /
        F.col("portfolio_expected_loss_ngn") *
        F.lit(100.0)
    )
    .withColumn(
        "expected_loss_rate_of_ead",
        F.col("stressed_expected_loss_ngn") /
        F.col("ead_ngn")
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
        .when(F.col("scenario") == "Moderate Macro Stress", 2)
        .when(F.col("scenario") == "Severe Macro Stress", 3)
        .otherwise(99),
        F.col("cluster")
    )
)

# ----------------------------------------------------------------
# Risk-band reconciliation against portfolio stress totals
# ----------------------------------------------------------------

risk_band_reconciliation = (
    stress_risk_band
    .groupBy("scenario")
    .agg(
        F.sum("transaction_count")
        .alias("aggregated_transaction_count"),

        F.sum("ead_ngn")
        .alias("aggregated_ead_ngn"),

        F.sum("stressed_pd_ead_exposure_ngn")
        .alias("aggregated_pd_ead_exposure_ngn"),

        F.sum("stressed_expected_loss_ngn")
        .alias("aggregated_expected_loss_ngn")
    )
    .join(
        stress_portfolio
        .select(
            "scenario",
            F.col("transaction_count")
            .alias("portfolio_transaction_count"),
            F.col("stressed_ead_ngn")
            .alias("portfolio_ead_ngn"),
            F.col("stressed_pd_ead_exposure_ngn")
            .alias("portfolio_pd_ead_exposure_ngn"),
            F.col("stressed_expected_loss_ngn")
            .alias("portfolio_expected_loss_ngn")
        ),
        on="scenario",
        how="inner"
    )
    .withColumn(
        "transaction_count_difference",
        F.col("aggregated_transaction_count") -
        F.col("portfolio_transaction_count")
    )
    .withColumn(
        "ead_difference_ngn",
        F.col("aggregated_ead_ngn") -
        F.col("portfolio_ead_ngn")
    )
    .withColumn(
        "pd_ead_difference_ngn",
        F.col("aggregated_pd_ead_exposure_ngn") -
        F.col("portfolio_pd_ead_exposure_ngn")
    )
    .withColumn(
        "expected_loss_difference_ngn",
        F.col("aggregated_expected_loss_ngn") -
        F.col("portfolio_expected_loss_ngn")
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
        .when(F.col("scenario") == "Moderate Macro Stress", 2)
        .when(F.col("scenario") == "Severe Macro Stress", 3)
        .otherwise(99)
    )
)

# ----------------------------------------------------------------
# Segment reconciliation against portfolio stress totals
# ----------------------------------------------------------------

segment_reconciliation = (
    stress_segment_summary
    .groupBy("scenario")
    .agg(
        F.sum("transaction_count")
        .alias("aggregated_transaction_count"),

        F.sum("ead_ngn")
        .alias("aggregated_ead_ngn"),

        F.sum("stressed_pd_ead_exposure_ngn")
        .alias("aggregated_pd_ead_exposure_ngn"),

        F.sum("stressed_expected_loss_ngn")
        .alias("aggregated_expected_loss_ngn")
    )
    .join(
        stress_portfolio
        .select(
            "scenario",
            F.col("transaction_count")
            .alias("portfolio_transaction_count"),
            F.col("stressed_ead_ngn")
            .alias("portfolio_ead_ngn"),
            F.col("stressed_pd_ead_exposure_ngn")
            .alias("portfolio_pd_ead_exposure_ngn"),
            F.col("stressed_expected_loss_ngn")
            .alias("portfolio_expected_loss_ngn")
        ),
        on="scenario",
        how="inner"
    )
    .withColumn(
        "transaction_count_difference",
        F.col("aggregated_transaction_count") -
        F.col("portfolio_transaction_count")
    )
    .withColumn(
        "ead_difference_ngn",
        F.col("aggregated_ead_ngn") -
        F.col("portfolio_ead_ngn")
    )
    .withColumn(
        "pd_ead_difference_ngn",
        F.col("aggregated_pd_ead_exposure_ngn") -
        F.col("portfolio_pd_ead_exposure_ngn")
    )
    .withColumn(
        "expected_loss_difference_ngn",
        F.col("aggregated_expected_loss_ngn") -
        F.col("portfolio_expected_loss_ngn")
    )
    .orderBy(
        F.when(F.col("scenario") == "Baseline", 1)
        .when(F.col("scenario") == "Moderate Macro Stress", 2)
        .when(F.col("scenario") == "Severe Macro Stress", 3)
        .otherwise(99)
    )
)

# ----------------------------------------------------------------
# Reconciliation quality gate
# ----------------------------------------------------------------

risk_band_reconciliation_invalid = (
    risk_band_reconciliation
    .filter(
        (F.abs(F.col("transaction_count_difference")) > 0) |
        (F.abs(F.col("ead_difference_ngn")) > 0.10) |
        (F.abs(F.col("pd_ead_difference_ngn")) > 0.10) |
        (F.abs(F.col("expected_loss_difference_ngn")) > 0.10)
    )
    .count()
)

segment_reconciliation_invalid = (
    segment_reconciliation
    .filter(
        (F.abs(F.col("transaction_count_difference")) > 0) |
        (F.abs(F.col("ead_difference_ngn")) > 0.10) |
        (F.abs(F.col("pd_ead_difference_ngn")) > 0.10) |
        (F.abs(F.col("expected_loss_difference_ngn")) > 0.10)
    )
    .count()
)

stress_aggregation_quality_schema = T.StructType([
    T.StructField("check", T.StringType(), False),
    T.StructField("passed", T.BooleanType(), False),
    T.StructField("observed_value", T.DoubleType(), False)
])

stress_aggregation_quality_rows = [
    (
        "Risk-band stress aggregation reconciles to portfolio",
        risk_band_reconciliation_invalid == 0,
        float(risk_band_reconciliation_invalid)
    ),
    (
        "Segment stress aggregation reconciles to portfolio",
        segment_reconciliation_invalid == 0,
        float(segment_reconciliation_invalid)
    ),
    (
        "Exactly five risk bands represented per scenario",
        stress_risk_band
        .groupBy("scenario")
        .agg(F.countDistinct("risk_band").alias("band_count"))
        .filter(F.col("band_count") != 5)
        .count() == 0,
        float(
            stress_risk_band
            .select("risk_band")
            .distinct()
            .count()
        )
    ),
    (
        "Exactly two customer segments represented per scenario",
        stress_segment_summary
        .groupBy("scenario")
        .agg(F.countDistinct("segment_label").alias("segment_count"))
        .filter(F.col("segment_count") != 2)
        .count() == 0,
        float(
            stress_segment_summary
            .select("segment_label")
            .distinct()
            .count()
        )
    )
]

stress_aggregation_quality = spark.createDataFrame(
    stress_aggregation_quality_rows,
    schema=stress_aggregation_quality_schema
)

display(stress_aggregation_quality)

if stress_aggregation_quality.filter(
    ~F.col("passed")
).count() > 0:
    raise ValueError(
        "Stress aggregation quality gate FAILED."
    )

print("Stress aggregation quality gate: PASS")

# ----------------------------------------------------------------
# Save risk-band stress layer
# ----------------------------------------------------------------

(
    stress_risk_band
    .drop(
        "portfolio_ead_ngn",
        "portfolio_expected_loss_ngn"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(STRESS_RISK_BAND_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Save segment stress layer
# ----------------------------------------------------------------

(
    stress_segment_summary
    .drop(
        "portfolio_ead_ngn",
        "portfolio_expected_loss_ngn"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(STRESS_SEGMENT_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Display key outputs
# ----------------------------------------------------------------

print("\nRisk-band stress impact:")
display(
    stress_risk_band.select(
        "scenario",
        "risk_band",
        "risk_band_order",
        "ead_ngn",
        "ead_share_pct",
        "average_stressed_pd",
        "stressed_expected_loss_ngn",
        "expected_loss_share_pct",
        "expected_loss_rate_of_ead"
    )
)

print("\nCustomer-segment stress impact:")
display(
    stress_segment_summary.select(
        "scenario",
        "cluster",
        "segment_label",
        "ead_ngn",
        "ead_share_pct",
        "average_stressed_pd",
        "stressed_expected_loss_ngn",
        "expected_loss_share_pct",
        "expected_loss_rate_of_ead"
    )
)

print("\nRisk-band reconciliation:")
display(risk_band_reconciliation)

print("\nSegment reconciliation:")
display(segment_reconciliation)

print(
    f"\nRisk-band stress output saved to: "
    f"{STRESS_RISK_BAND_OUTPUT_PATH}"
)

print(
    f"Segment stress output saved to: "
    f"{STRESS_SEGMENT_OUTPUT_PATH}"
)

check,passed,observed_value
Risk-band stress aggregation reconciles to portfolio,true,0.0
Segment stress aggregation reconciles to portfolio,true,0.0
Exactly five risk bands represented per scenario,true,5.0
Exactly two customer segments represented per scenario,true,2.0


Stress aggregation quality gate: PASS

Risk-band stress impact:


scenario,risk_band,risk_band_order,ead_ngn,ead_share_pct,average_stressed_pd,stressed_expected_loss_ngn,expected_loss_share_pct,expected_loss_rate_of_ead
Baseline,Very Low,1,4.996942780677544E9,14.997766439557344,0.02363777971730816,4.725124280495014E7,3.5908747762261113,0.009456030392756122
Baseline,Low,2,5.310990428719664E9,15.940345428942809,0.034710437086536185,7.438557372810112E7,5.652957775480206,0.014005970209596758
Baseline,Moderate,3,8.160189383352924E9,24.491898315771447,0.05172307428834527,1.7033720251392737E8,12.944835472054548,0.020874172707489033
Baseline,High,4,6.462766710921518E9,19.39727348060701,0.07877612773939319,2.0632231986963487E8,15.679537091767488,0.03192476675987204
Baseline,Very High,5,8.387023739018655E9,25.172716335120636,0.21123131560202987,8.175736300887955E8,62.13179488447047,0.09748078168483375
Moderate Macro Stress,Very Low,1,4.996942780677537E9,14.997766439557358,0.02954722464663475,8.121307357100835E7,3.590874776226078,0.016252552237549667
Moderate Macro Stress,Low,2,5.310990428719649E9,15.9403454289428,0.043388046358171076,1.2785020484516972E8,5.652957775479953,0.024072761297743724
Moderate Macro Stress,Moderate,3,8.1601893833529215E9,24.491898315771497,0.06465384286043108,2.9276706682081634E8,12.944835472054542,0.03587748434099724
Moderate Macro Stress,High,4,6.462766710921506E9,19.397273480607012,0.09847015967424193,3.546164872759391E8,15.679537091767468,0.05487069286853083
Moderate Macro Stress,Very High,5,8.387023739018657E9,25.172716335120697,0.26403914450253785,1.4052046767151308E9,62.13179488447026,0.16754509352080957



Customer-segment stress impact:


scenario,cluster,segment_label,ead_ngn,ead_share_pct,average_stressed_pd,stressed_expected_loss_ngn,expected_loss_share_pct,expected_loss_rate_of_ead
Baseline,0,Low-Engagement / Low-Exposure,2.4416848804018383E9,7.328444843688993,0.0755022963351042,9.019584478662878E7,6.854464871996556,0.03694000217251002
Baseline,1,Active / Higher-Exposure,3.087622816228844E10,92.67155515631018,0.08044323485951266,1.2256741242187805E9,93.1455351280023,0.03969636827971729
Moderate Macro Stress,0,Low-Engagement / Low-Exposure,2.4416848804018383E9,7.3284448436890095,0.0943778704188804,1.5502410822701854E8,6.854464871996481,0.06349062873400173
Moderate Macro Stress,1,Active / Higher-Exposure,3.0876228162288445E10,92.6715551563104,0.10055404357439081,2.1066274010010405E9,93.14553512800158,0.06822813298076445
Severe Macro Stress,0,Low-Engagement / Low-Exposure,2.4416848804018383E9,7.328444843688979,0.11325344450265637,2.3676409256490082E8,6.854464871996627,0.09696750570283892
Severe Macro Stress,1,Active / Higher-Exposure,3.087622816228844E10,92.67155515631,0.12066485228926895,3.217394576074324E9,93.1455351280039,0.1042029667342587



Risk-band reconciliation:


scenario,aggregated_transaction_count,aggregated_ead_ngn,aggregated_pd_ead_exposure_ngn,aggregated_expected_loss_ngn,portfolio_transaction_count,portfolio_ead_ngn,portfolio_pd_ead_exposure_ngn,portfolio_expected_loss_ngn,transaction_count_difference,ead_difference_ngn,pd_ead_difference_ngn,expected_loss_difference_ngn
Baseline,666246,3.33179130426903E10,3.2896749225135145E9,1.315869969005409E9,666246,3.331791304269064E10,3.2896749225134406E9,1.3158699690053992E9,0,-3.39508056640625E-4,7.390975952148438E-5,9.775161743164062E-6
Moderate Macro Stress,666246,3.331791304269027E10,4.1120936531419153E9,2.2616515092280645E9,666246,3.33179130426906E10,4.1120936531418667E9,2.261651509228124E9,0,-3.31878662109375E-4,4.863739013671875E-5,-5.9604644775390625E-5
Severe Macro Stress,666246,3.3317913042690292E10,4.934512383770298E9,3.454158668639221E9,666246,3.3317913042690617E10,4.934512383770449E9,3.4541586686391807E9,0,-3.24249267578125E-4,-1.506805419921875E-4,4.0531158447265625E-5



Segment reconciliation:


scenario,aggregated_transaction_count,aggregated_ead_ngn,aggregated_pd_ead_exposure_ngn,aggregated_expected_loss_ngn,portfolio_transaction_count,portfolio_ead_ngn,portfolio_pd_ead_exposure_ngn,portfolio_expected_loss_ngn,transaction_count_difference,ead_difference_ngn,pd_ead_difference_ngn,expected_loss_difference_ngn
Baseline,666246,3.331791304269027E10,3.289674922513541E9,1.315869969005412E9,666246,3.331791304269064E10,3.2896749225134406E9,1.3158699690053992E9,0,-3.70025634765625E-4,1.0061264038085938E-4,1.2874603271484375E-5
Moderate Macro Stress,666246,3.3317913042690266E10,4.112093653141915E9,2.261651509228059E9,666246,3.33179130426906E10,4.1120936531418667E9,2.261651509228124E9,0,-3.35693359375E-4,4.8160552978515625E-5,-6.532669067382812E-5
Severe Macro Stress,666246,3.331791304269026E10,4.934512383770307E9,3.454158668639219E9,666246,3.3317913042690617E10,4.934512383770449E9,3.4541586686391807E9,0,-3.54766845703125E-4,-1.4209747314453125E-4,3.814697265625E-5



Risk-band stress output saved to: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_risk_band
Segment stress output saved to: /Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_segment


## PD × LGD Sensitivity Analysis

Because both default incidence and loss severity are uncertain under stress, a two-dimensional sensitivity analysis is performed.

PD multipliers range from 1.00× to 1.50×, while LGD assumptions range from 40% to 70%, producing 49 combinations.

The sensitivity grid is intended to show how expected loss responds to alternative assumptions and to separate the contribution of PD deterioration from loss-severity assumptions.

The grid is a scenario analysis and should not be interpreted as 49 independent forecasts.

In [0]:
# ================================================================
# PD x LGD Sensitivity Analysis
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ----------------------------------------------------------------
# Paths
# ----------------------------------------------------------------

PORTFOLIO_RISK_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk"
)

SENSITIVITY_LONG_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_long"
)

SENSITIVITY_MATRIX_OUTPUT_PATH = (
    "/Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_matrix"
)

# ----------------------------------------------------------------
# Load authoritative portfolio risk layer
# ----------------------------------------------------------------

portfolio_risk_sensitivity = (
    spark.read
    .format("delta")
    .load(PORTFOLIO_RISK_PATH)
    .select(
        "transaction_id",
        "customer_id",
        "portfolio_pd",
        "ead_ngn"
    )
)

# ----------------------------------------------------------------
# Validate population
# ----------------------------------------------------------------

sensitivity_transaction_count = (
    portfolio_risk_sensitivity.count()
)

sensitivity_customer_count = (
    portfolio_risk_sensitivity
    .select("customer_id")
    .distinct()
    .count()
)

if sensitivity_transaction_count != 666246:
    raise ValueError(
        f"Unexpected sensitivity population. "
        f"Expected 666246 transactions, found "
        f"{sensitivity_transaction_count}."
    )

# ----------------------------------------------------------------
# Define sensitivity assumptions
# ----------------------------------------------------------------

pd_multiplier_schema = T.StructType([
    T.StructField("pd_multiplier", T.DoubleType(), False)
])

lgd_schema = T.StructType([
    T.StructField("lgd", T.DoubleType(), False)
])

pd_multiplier_values = [
    (1.00,),
    (1.10,),
    (1.20,),
    (1.25,),
    (1.30,),
    (1.40,),
    (1.50,)
]

lgd_values = [
    (0.40,),
    (0.45,),
    (0.50,),
    (0.55,),
    (0.60,),
    (0.65,),
    (0.70,)
]

pd_multiplier_df = spark.createDataFrame(
    pd_multiplier_values,
    schema=pd_multiplier_schema
)

lgd_df = spark.createDataFrame(
    lgd_values,
    schema=lgd_schema
)

# ----------------------------------------------------------------
# Generate 49 sensitivity combinations
# ----------------------------------------------------------------

sensitivity_grid = (
    pd_multiplier_df
    .crossJoin(lgd_df)
)

sensitivity_combination_count = (
    sensitivity_grid.count()
)

if sensitivity_combination_count != 49:
    raise ValueError(
        f"Expected 49 PD x LGD combinations, "
        f"found {sensitivity_combination_count}."
    )

# ----------------------------------------------------------------
# Calculate portfolio sensitivity results
#
# Stressed PD:
# min(base portfolio PD x PD multiplier, 1)
#
# Expected Loss:
# stressed PD x EAD x LGD
# ----------------------------------------------------------------

portfolio_sensitivity = (
    portfolio_risk_sensitivity
    .crossJoin(F.broadcast(sensitivity_grid))
    .withColumn(
        "stressed_pd",
        F.least(
            F.col("portfolio_pd") *
            F.col("pd_multiplier"),
            F.lit(1.0)
        )
    )
    .withColumn(
        "sensitivity_expected_loss_ngn",
        F.col("stressed_pd") *
        F.col("ead_ngn") *
        F.col("lgd")
    )
)

# ----------------------------------------------------------------
# Aggregate each sensitivity combination
# ----------------------------------------------------------------

sensitivity_long = (
    portfolio_sensitivity
    .groupBy(
        "pd_multiplier",
        "lgd"
    )
    .agg(
        F.count("*").alias(
            "transaction_count"
        ),

        F.countDistinct(
            "customer_id"
        ).alias(
            "customer_count"
        ),

        F.sum(
            "ead_ngn"
        ).alias(
            "ead_ngn"
        ),

        F.avg(
            "stressed_pd"
        ).alias(
            "transaction_weighted_pd"
        ),

        (
            F.sum(
                F.col("stressed_pd") *
                F.col("ead_ngn")
            )
            /
            F.sum("ead_ngn")
        ).alias(
            "exposure_weighted_pd"
        ),

        F.sum(
            F.col("stressed_pd") *
            F.col("ead_ngn")
        ).alias(
            "pd_ead_exposure_ngn"
        ),

        F.sum(
            "sensitivity_expected_loss_ngn"
        ).alias(
            "expected_loss_ngn"
        )
    )
    .withColumn(
        "expected_loss_rate_of_ead",
        F.col("expected_loss_ngn") /
        F.col("ead_ngn")
    )
)

# ----------------------------------------------------------------
# Baseline reference
# ----------------------------------------------------------------

baseline_el = (
    portfolio_risk_sensitivity
    .agg(
        F.sum(
            F.col("portfolio_pd") *
            F.col("ead_ngn") *
            F.lit(0.40)
        ).alias(
            "baseline_el"
        )
    )
    .collect()[0]["baseline_el"]
)

baseline_el = float(baseline_el)

# ----------------------------------------------------------------
# Incremental sensitivity impact
# ----------------------------------------------------------------

sensitivity_long = (
    sensitivity_long
    .withColumn(
        "baseline_expected_loss_ngn",
        F.lit(baseline_el)
    )
    .withColumn(
        "incremental_expected_loss_ngn",
        F.col("expected_loss_ngn") -
        F.lit(baseline_el)
    )
    .withColumn(
        "expected_loss_multiple",
        F.col("expected_loss_ngn") /
        F.lit(baseline_el)
    )
    .withColumn(
        "expected_loss_increase_pct",
        (
            F.col("expected_loss_ngn") /
            F.lit(baseline_el) -
            F.lit(1.0)
        ) *
        F.lit(100.0)
    )
    .orderBy(
        "pd_multiplier",
        "lgd"
    )
)

# ----------------------------------------------------------------
# Sensitivity quality checks
# ----------------------------------------------------------------

invalid_pd_count = (
    sensitivity_long
    .filter(
        (F.col("transaction_count") != sensitivity_transaction_count) |
        (F.col("customer_count") != sensitivity_customer_count)
    )
    .count()
)

invalid_ead_count = (
    sensitivity_long
    .filter(
        F.abs(
            F.col("ead_ngn") -
            F.lit(
                portfolio_risk_sensitivity
                .agg(F.sum("ead_ngn"))
                .collect()[0][0]
            )
        ) > F.lit(0.10)
    )
    .count()
)

invalid_expected_loss_count = (
    sensitivity_long
    .filter(
        F.col("expected_loss_ngn") < 0
    )
    .count()
)

# ----------------------------------------------------------------
# Check baseline sensitivity point
# ----------------------------------------------------------------

baseline_sensitivity_row = (
    sensitivity_long
    .filter(
        (F.col("pd_multiplier") == 1.00) &
        (F.col("lgd") == 0.40)
    )
    .select(
        "expected_loss_ngn"
    )
    .collect()
)

if len(baseline_sensitivity_row) != 1:
    raise ValueError(
        "Baseline sensitivity combination was not found."
    )

baseline_sensitivity_el = float(
    baseline_sensitivity_row[0]["expected_loss_ngn"]
)

baseline_difference = (
    baseline_sensitivity_el -
    baseline_el
)

# ----------------------------------------------------------------
# Check monotonicity
#
# Higher PD multiplier or higher LGD must not reduce EL.
# ----------------------------------------------------------------

pd_monotonicity_violations = (
    sensitivity_long
    .groupBy("lgd")
    .agg(
        F.min("expected_loss_ngn").alias("min_el"),
        F.max("expected_loss_ngn").alias("max_el")
    )
    .filter(
        F.col("max_el") < F.col("min_el")
    )
    .count()
)

lgd_monotonicity_violations = (
    sensitivity_long
    .groupBy("pd_multiplier")
    .agg(
        F.min("expected_loss_ngn").alias("min_el"),
        F.max("expected_loss_ngn").alias("max_el")
    )
    .filter(
        F.col("max_el") < F.col("min_el")
    )
    .count()
)

sensitivity_quality_schema = T.StructType([
    T.StructField("check", T.StringType(), False),
    T.StructField("passed", T.BooleanType(), False),
    T.StructField("observed_value", T.DoubleType(), False)
])

sensitivity_quality_rows = [
    (
        "Exactly 49 PD x LGD combinations",
        sensitivity_combination_count == 49,
        float(sensitivity_combination_count)
    ),
    (
        "Every combination covers the full transaction population",
        invalid_pd_count == 0,
        float(invalid_pd_count)
    ),
    (
        "Every combination covers the full portfolio EAD",
        invalid_ead_count == 0,
        float(invalid_ead_count)
    ),
    (
        "Expected loss values are non-negative",
        invalid_expected_loss_count == 0,
        float(invalid_expected_loss_count)
    ),
    (
        "Baseline sensitivity reconciles to portfolio EL",
        abs(baseline_difference) <= 0.10,
        float(baseline_difference)
    ),
    (
        "Expected loss is non-decreasing with PD multiplier",
        pd_monotonicity_violations == 0,
        float(pd_monotonicity_violations)
    ),
    (
        "Expected loss is non-decreasing with LGD",
        lgd_monotonicity_violations == 0,
        float(lgd_monotonicity_violations)
    )
]

sensitivity_quality = spark.createDataFrame(
    sensitivity_quality_rows,
    schema=sensitivity_quality_schema
)

display(sensitivity_quality)

if sensitivity_quality.filter(
    ~F.col("passed")
).count() > 0:
    raise ValueError(
        "PD x LGD sensitivity quality gate FAILED."
    )

print(
    "PD x LGD sensitivity quality gate: PASS"
)

# ----------------------------------------------------------------
# Save long-form sensitivity table
# ----------------------------------------------------------------

(
    sensitivity_long
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SENSITIVITY_LONG_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Create presentation-friendly sensitivity matrix
# ----------------------------------------------------------------

sensitivity_matrix = (
    sensitivity_long
    .groupBy("pd_multiplier")
    .pivot("lgd")
    .agg(
        F.first("expected_loss_ngn")
    )
    .orderBy("pd_multiplier")
)

# Rename LGD columns for readability
for lgd_value in lgd_values:
    lgd_number = float(lgd_value[0])

    old_name = str(lgd_number)

    new_name = (
        f"LGD_{int(round(lgd_number * 100))}pct_EL_NGN"
    )

    if old_name in sensitivity_matrix.columns:
        sensitivity_matrix = sensitivity_matrix.withColumnRenamed(
            old_name,
            new_name
        )

# ----------------------------------------------------------------
# Save sensitivity matrix
# ----------------------------------------------------------------

(
    sensitivity_matrix
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SENSITIVITY_MATRIX_OUTPUT_PATH)
)

# ----------------------------------------------------------------
# Display results
# ----------------------------------------------------------------

print("\n49-row PD x LGD sensitivity table:")

display(
    sensitivity_long.select(
        "pd_multiplier",
        "lgd",
        "expected_loss_ngn",
        "incremental_expected_loss_ngn",
        "expected_loss_multiple",
        "expected_loss_increase_pct",
        "expected_loss_rate_of_ead"
    )
)

print("\nPD x LGD sensitivity matrix:")

display(sensitivity_matrix)

print(
    f"\nLong-form sensitivity output saved to: "
    f"{SENSITIVITY_LONG_OUTPUT_PATH}"
)

print(
    f"Sensitivity matrix saved to: "
    f"{SENSITIVITY_MATRIX_OUTPUT_PATH}"
)

check,passed,observed_value
Exactly 49 PD x LGD combinations,true,49.0
Every combination covers the full transaction population,true,0.0
Every combination covers the full portfolio EAD,true,0.0
Expected loss values are non-negative,true,0.0
Baseline sensitivity reconciles to portfolio EL,true,0.0
Expected loss is non-decreasing with PD multiplier,true,0.0
Expected loss is non-decreasing with LGD,true,0.0


PD x LGD sensitivity quality gate: PASS

49-row PD x LGD sensitivity table:


pd_multiplier,lgd,expected_loss_ngn,incremental_expected_loss_ngn,expected_loss_multiple,expected_loss_increase_pct,expected_loss_rate_of_ead
1.0,0.4,1.3158699690054004E9,4.76837158203125E-7,1.0000000000000004,4.440892098500626E-14,0.039494369509859795
1.0,0.45,1.4803537151310966E9,1.6448374612569666E8,1.1250000000000164,12.500000000001643,0.044431165698592866
1.0,0.5,1.6448374612567205E9,3.289674922513206E8,1.2499999999999776,24.99999999999776,0.04936796188732379
1.0,0.55,1.8093212073825045E9,4.934512383771045E8,1.3750000000000604,37.50000000000604,0.05430475807605956
1.0,0.6,1.973804953508073E9,6.579349845026731E8,1.4999999999999796,49.999999999997954,0.059241554264788845
1.0,0.65,2.138288699633757E9,8.224187306283572E8,1.6249999999999865,62.49999999999864,0.06417835045352155
1.0,0.7,2.3027724457595534E9,9.869024767541535E8,1.7500000000000786,75.00000000000786,0.06911514664225767
1.1,0.4,1.4474569659059653E9,1.3158699690056539E8,1.1000000000000194,10.00000000000194,0.0434438064608465
1.1,0.45,1.6283890866441607E9,3.125191176387608E8,1.2374999999999836,23.749999999998362,0.048874282268450836
1.1,0.5,1.8093212073825E9,4.934512383771E8,1.375000000000057,37.500000000005706,0.054304758076059514



PD x LGD sensitivity matrix:


pd_multiplier,LGD_40pct_EL_NGN,LGD_45pct_EL_NGN,LGD_50pct_EL_NGN,LGD_55pct_EL_NGN,LGD_60pct_EL_NGN,LGD_65pct_EL_NGN,LGD_70pct_EL_NGN
1.0,1.3158699690054004E9,1.4803537151310966E9,1.6448374612567205E9,1.8093212073825045E9,1.973804953508073E9,2.138288699633757E9,2.3027724457595534E9
1.1,1.4474569659059653E9,1.6283890866441607E9,1.8093212073825E9,1.9902533281208136E9,2.171185448858983E9,2.352117569597149E9,2.5330496903353553E9
1.2,1.5790439628064716E9,1.7764244581572196E9,1.9738049535080693E9,2.1711854488589725E9,2.36856594420988E9,2.565946439560523E9,2.7633269349114013E9
1.25,1.6448374612567213E9,1.8504421439138944E9,2.056046826570934E9,2.261651509228119E9,2.467256191885223E9,2.672860874542224E9,2.8784655571994033E9
1.3,1.7106309597070258E9,1.9244598296704006E9,2.1382886996337488E9,2.352117569597162E9,2.5659464395605197E9,2.7797753095238786E9,2.993604179487436E9
1.4,1.8422179566076312E9,2.0724952011835256E9,2.302772445759552E9,2.5330496903353715E9,2.7633269349114075E9,2.993604179487431E9,3.223881424063299E9
1.5,1.9738049535080674E9,2.220530572696635E9,2.4672561918852267E9,2.7139818110737786E9,2.9607074302621875E9,3.207433049450712E9,3.454158668639178E9



Long-form sensitivity output saved to: /Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_long
Sensitivity matrix saved to: /Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_matrix


## Portfolio Risk Engine Quality Gate

The final quality gate validates the complete portfolio-risk pipeline before downstream business translation.

The checks cover:

- Portfolio population and transaction uniqueness
- PD validity
- EAD validity
- PD × EAD reconciliation
- LGD scenario completeness
- Expected-loss reconciliation
- Risk-band completeness
- Macro history coverage
- Macro stress scenario completeness
- Stress aggregation reconciliation
- PD × LGD sensitivity completeness
- Baseline sensitivity reconciliation

The Portfolio Risk Engine is frozen only if all quality checks pass.

In [0]:
# ================================================================
# FINAL PORTFOLIO RISK ENGINE QUALITY GATE
# ================================================================
#
# This block performs an end-to-end audit of Notebook 08 outputs.
# It does not introduce new modelling assumptions.
# ================================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# ----------------------------------------------------------------
# Authoritative output paths
# ----------------------------------------------------------------

OUTPUT_PATHS = {
    "portfolio_pd":
        "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_pd",

    "portfolio_risk_bands":
        "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk_bands",

    "portfolio_ead":
        "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_ead",

    "portfolio_risk":
        "/Volumes/workspace/default/bnpl_raw/bnpl_portfolio_risk",

    "expected_loss":
        "/Volumes/workspace/default/bnpl_raw/bnpl_expected_loss",

    "expected_loss_portfolio":
        "/Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_portfolio_summary",

    "expected_loss_risk_band":
        "/Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_risk_band",

    "expected_loss_segment":
        "/Volumes/workspace/default/bnpl_raw/bnpl_expected_loss_segment",

    "macro_context":
        "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_context",

    "macro_stress_transactions":
        "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_transactions",

    "macro_stress_portfolio":
        "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_portfolio_summary",

    "macro_stress_risk_band":
        "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_risk_band",

    "macro_stress_segment":
        "/Volumes/workspace/default/bnpl_raw/bnpl_macro_stress_segment",

    "pd_lgd_sensitivity":
        "/Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_long",

    "pd_lgd_sensitivity_matrix":
        "/Volumes/workspace/default/bnpl_raw/bnpl_pd_lgd_sensitivity_matrix"
}

# ----------------------------------------------------------------
# Load outputs
# ----------------------------------------------------------------

pd_layer = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["portfolio_pd"])
)

risk_band_layer = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["portfolio_risk_bands"])
)

ead_layer = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["portfolio_ead"])
)

portfolio_risk_layer = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["portfolio_risk"])
)

expected_loss_layer = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["expected_loss"])
)

expected_loss_portfolio = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["expected_loss_portfolio"])
)

expected_loss_risk_band = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["expected_loss_risk_band"])
)

expected_loss_segment = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["expected_loss_segment"])
)

macro_context = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["macro_context"])
)

macro_stress_transactions = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["macro_stress_transactions"])
)

macro_stress_portfolio = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["macro_stress_portfolio"])
)

macro_stress_risk_band = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["macro_stress_risk_band"])
)

macro_stress_segment = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["macro_stress_segment"])
)

pd_lgd_sensitivity = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["pd_lgd_sensitivity"])
)

pd_lgd_sensitivity_matrix = (
    spark.read
    .format("delta")
    .load(OUTPUT_PATHS["pd_lgd_sensitivity_matrix"])
)

print("All Notebook 08 analytical outputs loaded successfully.")

# ----------------------------------------------------------------
# Fixed population reference
# ----------------------------------------------------------------

EXPECTED_TRANSACTIONS = 666246

# ----------------------------------------------------------------
# Portfolio reference metrics
# ----------------------------------------------------------------

portfolio_metrics = (
    portfolio_risk_layer
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("transaction_id").alias("unique_transaction_count"),
        F.countDistinct("customer_id").alias("customer_count"),
        F.sum("ead_ngn").alias("total_ead_ngn"),
        F.avg("portfolio_pd").alias("mean_pd"),
        F.sum(
            F.col("portfolio_pd") *
            F.col("ead_ngn")
        ).alias("pd_ead_exposure_ngn")
    )
    .collect()[0]
)

transaction_count = int(
    portfolio_metrics["transaction_count"]
)

unique_transaction_count = int(
    portfolio_metrics["unique_transaction_count"]
)

customer_count = int(
    portfolio_metrics["customer_count"]
)

total_ead = float(
    portfolio_metrics["total_ead_ngn"]
)

mean_pd = float(
    portfolio_metrics["mean_pd"]
)

pd_ead_exposure = float(
    portfolio_metrics["pd_ead_exposure_ngn"]
)

# ----------------------------------------------------------------
# PD checks
# ----------------------------------------------------------------

pd_invalid_count = (
    pd_layer
    .filter(
        F.col("portfolio_pd").isNull() |
        (F.col("portfolio_pd") < 0) |
        (F.col("portfolio_pd") > 1)
    )
    .count()
)

pd_transaction_count = pd_layer.count()

pd_unique_transaction_count = (
    pd_layer
    .select("transaction_id")
    .distinct()
    .count()
)

# ----------------------------------------------------------------
# EAD checks
# ----------------------------------------------------------------

ead_invalid_count = (
    ead_layer
    .filter(
        F.col("ead_ngn").isNull() |
        (F.col("ead_ngn") <= 0)
    )
    .count()
)

ead_transaction_count = ead_layer.count()

# ----------------------------------------------------------------
# PD x EAD reconciliation
# ----------------------------------------------------------------

risk_pd_ead = (
    portfolio_risk_layer
    .agg(
        F.sum(
            F.col("portfolio_pd") *
            F.col("ead_ngn")
        ).alias("calculated_pd_ead")
    )
    .collect()[0]["calculated_pd_ead"]
)

risk_pd_ead = float(risk_pd_ead)

pd_ead_difference = (
    risk_pd_ead -
    pd_ead_exposure
)

# ----------------------------------------------------------------
# LGD / EL checks
# ----------------------------------------------------------------

lgd_scenario_count = (
    expected_loss_layer
    .select("lgd_scenario")
    .distinct()
    .count()
)

lgd_values = [
    float(row["lgd"])
    for row in (
        expected_loss_layer
        .select("lgd")
        .distinct()
        .collect()
    )
]

lgd_valid = (
    lgd_scenario_count == 3 and
    all(0.0 <= value <= 1.0 for value in lgd_values)
)

expected_loss_transaction_count = (
    expected_loss_layer.count()
)

expected_loss_expected_rows = (
    EXPECTED_TRANSACTIONS * 3
)

# ----------------------------------------------------------------
# Baseline EL reconciliation
# ----------------------------------------------------------------

baseline_el_layer = (
    expected_loss_portfolio
    .filter(
        F.col("lgd_scenario") == "Baseline"
    )
    .select("expected_loss_ngn")
    .collect()
)

if len(baseline_el_layer) != 1:
    raise ValueError(
        "Baseline expected-loss result is missing or duplicated."
    )

baseline_el = float(
    baseline_el_layer[0]["expected_loss_ngn"]
)

independent_baseline_el = (
    pd_ead_exposure * 0.40
)

baseline_el_difference = (
    baseline_el -
    independent_baseline_el
)

# ----------------------------------------------------------------
# Risk-band checks
# ----------------------------------------------------------------

risk_band_count = (
    risk_band_layer
    .select("risk_band")
    .distinct()
    .count()
)

risk_band_transaction_count = (
    risk_band_layer
    .select("transaction_id")
    .distinct()
    .count()
)

risk_band_missing_count = (
    risk_band_layer
    .filter(
        F.col("risk_band").isNull()
    )
    .count()
)

# ----------------------------------------------------------------
# Macro checks
# ----------------------------------------------------------------

macro_observation_count = (
    macro_context
    .select("macro_date")
    .distinct()
    .count()
)

macro_min_date = (
    macro_context
    .agg(F.min("macro_date"))
    .collect()[0][0]
)

macro_max_date = (
    macro_context
    .agg(F.max("macro_date"))
    .collect()[0][0]
)

# ----------------------------------------------------------------
# Macro stress checks
# ----------------------------------------------------------------

macro_stress_scenario_count = (
    macro_stress_portfolio
    .select("scenario")
    .distinct()
    .count()
)

macro_stress_transaction_rows = (
    macro_stress_transactions.count()
)

expected_macro_stress_rows = (
    EXPECTED_TRANSACTIONS * 3
)

# ----------------------------------------------------------------
# Stress baseline reconciliation
# ----------------------------------------------------------------

stress_baseline_el = (
    macro_stress_portfolio
    .filter(
        F.col("scenario") == "Baseline"
    )
    .select("stressed_expected_loss_ngn")
    .collect()
)

if len(stress_baseline_el) != 1:
    raise ValueError(
        "Macro stress baseline EL is missing or duplicated."
    )

stress_baseline_el = float(
    stress_baseline_el[0]["stressed_expected_loss_ngn"]
)

stress_baseline_difference = (
    stress_baseline_el -
    baseline_el
)

# ----------------------------------------------------------------
# Stress aggregation checks
# ----------------------------------------------------------------

stress_risk_band_scenarios = (
    macro_stress_risk_band
    .select("scenario")
    .distinct()
    .count()
)

stress_segment_scenarios = (
    macro_stress_segment
    .select("scenario")
    .distinct()
    .count()
)

# ----------------------------------------------------------------
# Sensitivity checks
# ----------------------------------------------------------------

sensitivity_row_count = (
    pd_lgd_sensitivity.count()
)

sensitivity_pd_count = (
    pd_lgd_sensitivity
    .select("pd_multiplier")
    .distinct()
    .count()
)

sensitivity_lgd_count = (
    pd_lgd_sensitivity
    .select("lgd")
    .distinct()
    .count()
)

# ----------------------------------------------------------------
# Sensitivity baseline reconciliation
# ----------------------------------------------------------------

sensitivity_baseline = (
    pd_lgd_sensitivity
    .filter(
        (F.col("pd_multiplier") == 1.0) &
        (F.col("lgd") == 0.40)
    )
    .select("expected_loss_ngn")
    .collect()
)

if len(sensitivity_baseline) != 1:
    raise ValueError(
        "Baseline sensitivity point is missing or duplicated."
    )

sensitivity_baseline_el = float(
    sensitivity_baseline[0]["expected_loss_ngn"]
)

sensitivity_baseline_difference = (
    sensitivity_baseline_el -
    baseline_el
)

# ----------------------------------------------------------------
# Build final quality-gate table
# ----------------------------------------------------------------

quality_schema = T.StructType([
    T.StructField("check", T.StringType(), False),
    T.StructField("passed", T.BooleanType(), False),
    T.StructField("observed_value", T.DoubleType(), False)
])

quality_rows = [

    (
        "Portfolio transaction population = 666,246",
        transaction_count == EXPECTED_TRANSACTIONS,
        float(transaction_count)
    ),

    (
        "Portfolio transaction IDs are unique",
        unique_transaction_count == EXPECTED_TRANSACTIONS,
        float(unique_transaction_count)
    ),

    (
        "Portfolio customer population is non-zero",
        customer_count > 0,
        float(customer_count)
    ),

    (
        "Portfolio EAD is positive",
        total_ead > 0,
        float(total_ead)
    ),

    (
        "Portfolio PD values are within [0, 1]",
        pd_invalid_count == 0,
        float(pd_invalid_count)
    ),

    (
        "PD layer population matches portfolio",
        pd_transaction_count == EXPECTED_TRANSACTIONS,
        float(pd_transaction_count)
    ),

    (
        "PD layer transaction IDs are unique",
        pd_unique_transaction_count == EXPECTED_TRANSACTIONS,
        float(pd_unique_transaction_count)
    ),

    (
        "EAD values are positive and non-null",
        ead_invalid_count == 0,
        float(ead_invalid_count)
    ),

    (
        "EAD layer population matches portfolio",
        ead_transaction_count == EXPECTED_TRANSACTIONS,
        float(ead_transaction_count)
    ),

    (
        "PD x EAD reconciliation",
        abs(pd_ead_difference) <= 0.10,
        float(pd_ead_difference)
    ),

    (
        "Exactly three LGD scenarios",
        lgd_scenario_count == 3,
        float(lgd_scenario_count)
    ),

    (
        "LGD values are valid",
        lgd_valid,
        float(len(lgd_values))
    ),

    (
        "Expected-loss layer has 3 rows per transaction",
        expected_loss_transaction_count == expected_loss_expected_rows,
        float(expected_loss_transaction_count)
    ),

    (
        "Baseline EL reconciles to PD x EAD x LGD",
        abs(baseline_el_difference) <= 0.10,
        float(baseline_el_difference)
    ),

    (
        "Exactly five portfolio risk bands",
        risk_band_count == 5,
        float(risk_band_count)
    ),

    (
        "Risk-band transaction population matches portfolio",
        risk_band_transaction_count == EXPECTED_TRANSACTIONS,
        float(risk_band_transaction_count)
    ),

    (
        "No missing risk-band assignments",
        risk_band_missing_count == 0,
        float(risk_band_missing_count)
    ),

    (
        "Macro history contains 36 observations",
        macro_observation_count == 36,
        float(macro_observation_count)
    ),

    (
        "Macro history begins in January 2022",
        str(macro_min_date) == "2022-01-01",
        1.0 if str(macro_min_date) == "2022-01-01" else 0.0
    ),

    (
        "Macro history ends in December 2024",
        str(macro_max_date) == "2024-12-01",
        1.0 if str(macro_max_date) == "2024-12-01" else 0.0
    ),

    (
        "Exactly three macro stress scenarios",
        macro_stress_scenario_count == 3,
        float(macro_stress_scenario_count)
    ),

    (
        "Macro stress transaction population = 3 x portfolio",
        macro_stress_transaction_rows == expected_macro_stress_rows,
        float(macro_stress_transaction_rows)
    ),

    (
        "Macro stress baseline reconciles to baseline EL",
        abs(stress_baseline_difference) <= 0.10,
        float(stress_baseline_difference)
    ),

    (
        "Risk-band stress layer contains all scenarios",
        stress_risk_band_scenarios == 3,
        float(stress_risk_band_scenarios)
    ),

    (
        "Segment stress layer contains all scenarios",
        stress_segment_scenarios == 3,
        float(stress_segment_scenarios)
    ),

    (
        "Sensitivity grid contains 49 combinations",
        sensitivity_row_count == 49,
        float(sensitivity_row_count)
    ),

    (
        "Sensitivity grid contains 7 PD multipliers",
        sensitivity_pd_count == 7,
        float(sensitivity_pd_count)
    ),

    (
        "Sensitivity grid contains 7 LGD assumptions",
        sensitivity_lgd_count == 7,
        float(sensitivity_lgd_count)
    ),

    (
        "Sensitivity baseline reconciles to baseline EL",
        abs(sensitivity_baseline_difference) <= 0.10,
        float(sensitivity_baseline_difference)
    )
]

final_quality_gate = spark.createDataFrame(
    quality_rows,
    schema=quality_schema
)

# ----------------------------------------------------------------
# Display final quality gate
# ----------------------------------------------------------------

display(final_quality_gate)

# ----------------------------------------------------------------
# Final decision
# ----------------------------------------------------------------

failed_checks = (
    final_quality_gate
    .filter(~F.col("passed"))
    .count()
)

total_checks = final_quality_gate.count()
passed_checks = total_checks - failed_checks

print(
    f"\nQuality checks passed: "
    f"{passed_checks}/{total_checks}"
)

if failed_checks > 0:
    raise ValueError(
        f"PORTFOLIO RISK ENGINE QUALITY GATE FAILED: "
        f"{failed_checks} check(s) failed."
    )

print("\n" + "=" * 70)
print("PORTFOLIO RISK ENGINE QUALITY GATE: PASSED")
print("=" * 70)
print("\nNotebook 08 is ready to be frozen.")

All Notebook 08 analytical outputs loaded successfully.


check,passed,observed_value
"Portfolio transaction population = 666,246",true,666246.0
Portfolio transaction IDs are unique,true,666246.0
Portfolio customer population is non-zero,true,421457.0
Portfolio EAD is positive,true,3.3317913042690628E10
"Portfolio PD values are within [0, 1]",true,0.0
PD layer population matches portfolio,true,666246.0
PD layer transaction IDs are unique,true,666246.0
EAD values are positive and non-null,true,0.0
EAD layer population matches portfolio,true,666246.0
PD x EAD reconciliation,true,0.0



Quality checks passed: 29/29

PORTFOLIO RISK ENGINE QUALITY GATE: PASSED

Notebook 08 is ready to be frozen.
